<div style="direction: rtl; white-space: normal; line-height: 1;">

# Notebook 05 — Transcript Inventory

این Notebook مرحله‌ی شناسایی، طبقه‌بندی و پذیرش Transcriptها برای ورود به Pipeline است.

هدف این فایل هنوز استخراج نهایی متن یا ساخت Segmentهای RAG نیست.

در این مرحله باید مشخص شود:

- چه Transcriptهایی در پروژه وجود دارند.
- هر فایل مربوط به کدام برنامه، فصل یا قسمت است.
- کدام فایل Source اصلی است.
- آیا Duplicate یا نسخه‌ی جایگزین وجود دارد.
- ساختار گویندگان هر برنامه چگونه است.
- در کدام برنامه Speaker Attribution لازم است.
- کدام فایل‌ها برای Transcript Ingestion پذیرفته می‌شوند.
- Provenance و Hash هر فایل چیست.
- چه اطلاعاتی با اطمینان از خود فایل قابل اثبات است و چه اطلاعاتی بر اساس تأیید انسانی ثبت شده است.

اطلاعات انسانی تأییدشده درباره برنامه‌ها:

- `در جستجوی خوشبختی` — اجرای تک‌نفره دکتر کمیل رودی
- `تربیت مربی` — اجرای تک‌نفره دکتر کمیل رودی
- `ام کیو` — اجرای تک‌نفره دکتر کمیل رودی
- `فناپ` — اجرای تک‌نفره دکتر کمیل رودی
- `عصر شیرین` — دکتر رودی، مجری و حدود هشت شرکت‌کننده؛ دو فصل و در مجموع ۲۶ قسمت
- `خانه تو` — دکتر رودی و یک مجری
- `چهلستون` — برنامه رادیویی با دکتر رودی، مجری و مهمان‌های دیگر
- `کتاب باز` — دکتر رودی و مجری
- `یونی تهران` — برنامه مرتبط با دانشگاه تهران با دکتر رودی و مجری

در برنامه‌های تک‌نفره نیازی به تشخیص Speaker وجود ندارد.

در برنامه‌های چندنفره، صحبت مجری، مخاطب یا مهمان حذف نمی‌شود؛ زیرا سؤال، پاسخ و Context گفتگو برای درک صحیح صحبت دکتر رودی اهمیت دارد.

منبع اصلی دانش همچنان صحبت‌های دکتر کمیل رودی است، اما Question/Response Context نیز باید حفظ شود.

به‌خصوص روش پاسخ‌گویی دکتر رودی که در آن با پرسیدن سؤال و ایجاد قلاب ذهنی مخاطب را به پاسخ هدایت می‌کند، بخشی از ساختار گفتگو محسوب می‌شود و نباید با حذف Turnهای مخاطب از بین برود.

در این Notebook هیچ Speaker Name ناشناخته‌ای بدون Evidence حدس زده نمی‌شود.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1;">

### شروع Transcript Inventory و Source Discovery

اولین مرحله، شناسایی محیط پروژه و فایل‌های Transcript موجود روی Disk است.

در این مرحله:

- Project Root به‌صورت Dynamic پیدا می‌شود.
- اطلاعات انسانی تأییدشده درباره ۹ برنامه در حافظه ثبت می‌شود.
- فایل‌ها و مسیرهای احتمالی Transcript جستجو می‌شوند.
- نام برنامه‌ها با Filenameها تطبیق اولیه داده می‌شود.
- هنوز هیچ فایل Transcript پذیرفته یا رد نمی‌شود.
- هنوز Speaker Diarization انجام نمی‌شود.
- هنوز متن Transcript استخراج نمی‌شود.
- هیچ Artifact جدیدی ساخته نمی‌شود.

این Cell فقط `Start Gate + Source Discovery` است.

**معیار PASS:**

- Project Root معتبر پیدا شود.
- پوشه `data` وجود داشته باشد.
- Context انسانی هر ۹ برنامه ثبت شده باشد.
- حداقل یک Candidate Transcript Source روی Disk شناسایی شود.

اگر Sourceها در مسیر یا نام‌گذاری متفاوتی باشند، Gate باید بدون حدس این موضوع را آشکار کند تا در Cell بعدی بررسی شود.

</div>

In [1]:
# Code Cell 01

from pathlib import Path
import re
import unicodedata

def find_project_root(start_path):
    current = start_path.resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / ".git").exists()
            and (candidate / "notebooks" / "phase2").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root could not be identified."
    )

def normalize_fa_key(value):
    value = unicodedata.normalize("NFKC", str(value))
    value = (
        value
        .replace("ي", "ی")
        .replace("ك", "ک")
        .replace("\u200c", "")
        .lower()
    )

    return re.sub(
        r"[^0-9a-zآ-ی]+",
        "",
        value,
    )

project_root = find_project_root(Path.cwd())

notebook_05_path = (
    project_root
    / "notebooks"
    / "phase2"
    / "05_transcript_inventory.ipynb"
)

data_root = project_root / "data"

human_confirmed_program_context = {
    "در جستجوی خوشبختی": {
        "format": "SOLO",
        "primary_speaker": "Dr. Komeil Roudi",
        "speaker_attribution_required": False,
    },
    "تربیت مربی": {
        "format": "SOLO",
        "primary_speaker": "Dr. Komeil Roudi",
        "speaker_attribution_required": False,
    },
    "ام کیو": {
        "format": "SOLO",
        "primary_speaker": "Dr. Komeil Roudi",
        "speaker_attribution_required": False,
    },
    "فناپ": {
        "format": "SOLO",
        "primary_speaker": "Dr. Komeil Roudi",
        "speaker_attribution_required": False,
    },
    "عصر شیرین": {
        "format": "MULTI_SPEAKER",
        "primary_speaker": "Dr. Komeil Roudi",
        "speaker_attribution_required": True,
        "known_roles": [
            "Dr. Komeil Roudi",
            "host",
            "participants",
        ],
        "approx_participants": 8,
        "season_count": 2,
        "episode_count": 26,
    },
    "خانه تو": {
        "format": "TWO_SPEAKER",
        "primary_speaker": "Dr. Komeil Roudi",
        "speaker_attribution_required": True,
        "known_roles": [
            "Dr. Komeil Roudi",
            "host",
        ],
    },
    "چهلستون": {
        "format": "MULTI_SPEAKER_RADIO",
        "primary_speaker": "Dr. Komeil Roudi",
        "speaker_attribution_required": True,
        "known_roles": [
            "Dr. Komeil Roudi",
            "host",
            "guests",
        ],
    },
    "کتاب باز": {
        "format": "TWO_SPEAKER",
        "primary_speaker": "Dr. Komeil Roudi",
        "speaker_attribution_required": True,
        "known_roles": [
            "Dr. Komeil Roudi",
            "host",
        ],
    },
    "یونی تهران": {
        "format": "TWO_SPEAKER",
        "primary_speaker": "Dr. Komeil Roudi",
        "speaker_attribution_required": True,
        "known_roles": [
            "Dr. Komeil Roudi",
            "host",
        ],
        "institution_context": "University of Tehran",
    },
}

human_confirmed_transcript_policy = {
    "primary_knowledge_speaker": "Dr. Komeil Roudi",
    "preserve_interlocutor_context": True,
    "preserve_question_response_structure": True,
    "preserve_socratic_questioning_context": True,
    "infer_unknown_speaker_names": False,
    "solo_programs_require_diarization": False,
    "multi_speaker_programs_require_role_attribution": True,
}

program_keys = {
    program_name: normalize_fa_key(program_name)
    for program_name in human_confirmed_program_context
}

candidate_files = []
candidate_directories = set()

if data_root.exists():
    for path in data_root.rglob("*"):
        normalized_path = normalize_fa_key(
            str(path.relative_to(data_root))
        )

        transcript_path_hint = (
            "transcript" in normalized_path
            or "پیاده" in normalized_path
        )

        program_matches = [
            program_name
            for program_name, program_key in program_keys.items()
            if (
                program_key
                and program_key in normalized_path
            )
        ]

        if path.is_dir():
            if transcript_path_hint:
                candidate_directories.add(path)
            continue

        if transcript_path_hint or program_matches:
            candidate_files.append({
                "relative_path": str(
                    path.relative_to(project_root)
                ),
                "filename": path.name,
                "suffix": path.suffix.lower(),
                "program_matches": program_matches,
            })

candidate_files = sorted(
    candidate_files,
    key=lambda item: item["relative_path"],
)

candidate_directories = sorted(
    candidate_directories,
    key=lambda path: str(path),
)

start_gate_pass = (
    project_root.exists()
    and notebook_05_path.exists()
    and data_root.exists()
    and len(human_confirmed_program_context) == 9
    and len(candidate_files) > 0
)

print("NOTEBOOK 05 — TRANSCRIPT INVENTORY START GATE")
print("-" * 80)
print("Project root:", project_root)
print(
    "Notebook:",
    notebook_05_path.relative_to(project_root),
)
print(
    "Human-confirmed programs:",
    len(human_confirmed_program_context),
)
print(
    "Solo programs:",
    sum(
        not item["speaker_attribution_required"]
        for item in human_confirmed_program_context.values()
    ),
)
print(
    "Programs requiring speaker attribution:",
    sum(
        item["speaker_attribution_required"]
        for item in human_confirmed_program_context.values()
    ),
)

print("\nCANDIDATE TRANSCRIPT DIRECTORIES")
print("-" * 80)

if candidate_directories:
    for directory in candidate_directories:
        print(
            directory.relative_to(project_root)
        )
else:
    print("NONE")

print("\nCANDIDATE TRANSCRIPT FILES")
print("-" * 80)
print("Candidate files:", len(candidate_files))

for index, item in enumerate(candidate_files, start=1):
    print(
        f"[{index}] {item['relative_path']}"
    )
    print(
        "    Program matches:",
        item["program_matches"]
        if item["program_matches"]
        else "UNRESOLVED",
    )
    print(
        "    Extension:",
        item["suffix"] or "NO_EXTENSION",
    )

print()
print(
    "TRANSCRIPT INVENTORY START GATE:",
    "PASS" if start_gate_pass else "FAIL",
)

NOTEBOOK 05 — TRANSCRIPT INVENTORY START GATE
--------------------------------------------------------------------------------
Project root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Notebook: notebooks/phase2/05_transcript_inventory.ipynb
Human-confirmed programs: 9
Solo programs: 4
Programs requiring speaker attribution: 5

CANDIDATE TRANSCRIPT DIRECTORIES
--------------------------------------------------------------------------------
data/transcripts
data/transcripts/final
data/transcripts/final/asre_shirin
data/transcripts/final/asre_shirin/season-1
data/transcripts/final/asre_shirin/season-2
data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat
data/transcripts/final/dar-jostojo-final-C
data/transcripts/final/dar-jostojo-final-C/extracted
data/transcripts/final/dar-jostojo-final-C/extracted/candidate-differences
data/transcripts/final/dar-jostojo-final-C/extracted/missing-core
data/transcripts/final/dar-jostojo-final-C/extracted/supplemental

<div style="direction: rtl; white-space: normal; line-height: 1;">

### پاک‌سازی Scope و نگاشت Canonical برنامه‌های Transcript

Start Gate با موفقیت اجرا شد:

`TRANSCRIPT INVENTORY START GATE: PASS`

Discovery اولیه نشان داد که جستجوی گسترده روی کل پوشه `data` برای Inventory دقیق کافی نیست.

دو مسئله مشخص شد:

- فایل‌های Book به‌دلیل تطبیق نام `تربیت مربی` وارد Candidateها شده‌اند، در حالی که Transcript نیستند.
- چند برنامه در نام پوشه به‌صورت لاتین یا اختصاری ذخیره شده‌اند و بنابراین Matching مستقیم فارسی آن‌ها را شناسایی نکرده است.

از این مرحله به بعد Scope اصلی Transcript Inventory محدود می‌شود به:

`data/transcripts/final`

برای برنامه‌های تأییدشده، نگاشت Path به Canonical Program تعریف می‌شود:

- `asre_shirin` → `عصر شیرین`
- `dar-jostojo-final-C` → `در جستجوی خوشبختی`
- `khane-to` → `خانه تو`
- `mq` → `ام کیو`
- `tarbiat-morabi` → `تربیت مربی`
- `uni-tehran` → `یونی تهران`

پوشه مشترک:

`chehelstion_finup_ketabbaz_bankmellat`

چند مجموعه متفاوت را در خود نگه می‌دارد.

در این پوشه فقط فایل‌هایی که شواهد نامی روشن دارند به برنامه‌های تأییدشده متصل می‌شوند:

- `چهلستون`
- `فناپ`
- `کتاب باز`

فایل‌هایی مانند:

- بانک ملت
- اقیانوس آرام
- شراکت
- میزان

فعلاً با وضعیت:

`UNMAPPED_PROGRAM`

ثبت می‌شوند و بدون تأیید انسانی به هیچ برنامه‌ای نسبت داده نمی‌شوند.

همچنین فایل‌های داخل زیرپوشه `extracted` مربوط به مجموعه `در جستجوی خوشبختی` به‌عنوان Working / Review Material از Primary Transcript Files جدا می‌شوند و در این مرحله Source اصلی محسوب نمی‌شوند.

هدف این Cell ساخت یک Inventory تمیز از Candidateهای واقعی Transcript است.

**معیار PASS:**

- فقط فایل‌های زیر `data/transcripts/final` بررسی شوند.
- فایل‌های سیستمی مانند `.DS_Store` حذف شوند.
- ۹ برنامه Human-confirmed در صورت وجود روی Disk به Canonical Name نگاشت شوند.
- فایل‌های ناشناخته بدون حدس به‌عنوان `UNMAPPED_PROGRAM` باقی بمانند.
- فایل‌های Review/Extracted از Primary Transcriptها جدا شوند.

</div>

In [2]:
# Code Cell 02

from collections import Counter, defaultdict

transcript_final_root = (
    project_root
    / "data"
    / "transcripts"
    / "final"
)

directory_program_map = {
    "asre_shirin": "عصر شیرین",
    "dar-jostojo-final-C": "در جستجوی خوشبختی",
    "khane-to": "خانه تو",
    "mq": "ام کیو",
    "tarbiat-morabi": "تربیت مربی",
    "uni-tehran": "یونی تهران",
}

shared_directory_name = (
    "chehelstion_finup_ketabbaz_bankmellat"
)

def classify_shared_directory_file(filename):
    stem = Path(filename).stem.strip()

    if stem.startswith("چهلستون"):
        return "چهلستون"

    if stem == "فناپ":
        return "فناپ"

    if stem == "کتاب باز":
        return "کتاب باز"

    return "UNMAPPED_PROGRAM"

clean_transcript_inventory = []

for path in sorted(
    transcript_final_root.rglob("*")
):
    if not path.is_file():
        continue

    if path.name == ".DS_Store":
        continue

    relative_to_final = path.relative_to(
        transcript_final_root
    )

    parts = relative_to_final.parts

    if not parts:
        continue

    top_directory = parts[0]

    is_review_material = (
        top_directory == "dar-jostojo-final-C"
        and "extracted" in parts
    )

    if top_directory in directory_program_map:
        canonical_program = directory_program_map[
            top_directory
        ]

    elif top_directory == shared_directory_name:
        canonical_program = (
            classify_shared_directory_file(
                path.name
            )
        )

    else:
        canonical_program = "UNMAPPED_PROGRAM"

    source_role = (
        "REVIEW_OR_WORKING_MATERIAL"
        if is_review_material
        else "PRIMARY_TRANSCRIPT_CANDIDATE"
    )

    clean_transcript_inventory.append({
        "relative_path": str(
            path.relative_to(project_root)
        ),
        "filename": path.name,
        "suffix": path.suffix.lower(),
        "top_directory": top_directory,
        "canonical_program": canonical_program,
        "source_role": source_role,
    })

primary_transcript_candidates = [
    record
    for record in clean_transcript_inventory
    if (
        record["source_role"]
        == "PRIMARY_TRANSCRIPT_CANDIDATE"
    )
]

review_materials = [
    record
    for record in clean_transcript_inventory
    if (
        record["source_role"]
        == "REVIEW_OR_WORKING_MATERIAL"
    )
]

mapped_primary_records = [
    record
    for record in primary_transcript_candidates
    if (
        record["canonical_program"]
        != "UNMAPPED_PROGRAM"
    )
]

unmapped_primary_records = [
    record
    for record in primary_transcript_candidates
    if (
        record["canonical_program"]
        == "UNMAPPED_PROGRAM"
    )
]

program_file_counts = Counter(
    record["canonical_program"]
    for record in mapped_primary_records
)

files_by_program = defaultdict(list)

for record in mapped_primary_records:
    files_by_program[
        record["canonical_program"]
    ].append(record["relative_path"])

confirmed_programs_on_disk = sorted(
    program
    for program in human_confirmed_program_context
    if program_file_counts.get(program, 0) > 0
)

missing_confirmed_programs = sorted(
    set(human_confirmed_program_context)
    - set(confirmed_programs_on_disk)
)

scope_cleanup_pass = (
    transcript_final_root.exists()
    and all(
        record["relative_path"].startswith(
            "data/transcripts/final/"
        )
        for record in clean_transcript_inventory
    )
    and len(confirmed_programs_on_disk) == 9
)

print("TRANSCRIPT INVENTORY — CLEAN SCOPE")
print("-" * 80)
print(
    "Transcript root:",
    transcript_final_root.relative_to(project_root),
)
print(
    "All non-system files:",
    len(clean_transcript_inventory),
)
print(
    "Primary transcript candidates:",
    len(primary_transcript_candidates),
)
print(
    "Review / working materials:",
    len(review_materials),
)
print(
    "Mapped primary files:",
    len(mapped_primary_records),
)
print(
    "Unmapped primary files:",
    len(unmapped_primary_records),
)

print("\nHUMAN-CONFIRMED PROGRAM COVERAGE")
print("-" * 80)

for program in human_confirmed_program_context:
    print(
        f"{program}:",
        program_file_counts.get(program, 0),
    )

print()
print(
    "Confirmed programs found:",
    len(confirmed_programs_on_disk),
    "/",
    len(human_confirmed_program_context),
)
print(
    "Missing confirmed programs:",
    missing_confirmed_programs
    if missing_confirmed_programs
    else "NONE",
)

print("\nUNMAPPED PRIMARY FILES")
print("-" * 80)

if unmapped_primary_records:
    for record in unmapped_primary_records:
        print(
            " -",
            record["relative_path"],
        )
else:
    print("NONE")

print("\nREVIEW / WORKING MATERIAL SUMMARY")
print("-" * 80)
print(
    "Files excluded from primary candidates:",
    len(review_materials),
)

print()
print(
    "TRANSCRIPT INVENTORY SCOPE CLEANUP GATE:",
    "PASS" if scope_cleanup_pass else "FAIL",
)

TRANSCRIPT INVENTORY — CLEAN SCOPE
--------------------------------------------------------------------------------
Transcript root: data/transcripts/final
All non-system files: 213
Primary transcript candidates: 183
Review / working materials: 30
Mapped primary files: 168
Unmapped primary files: 15

HUMAN-CONFIRMED PROGRAM COVERAGE
--------------------------------------------------------------------------------
در جستجوی خوشبختی: 29
تربیت مربی: 14
ام کیو: 8
فناپ: 1
عصر شیرین: 26
خانه تو: 13
چهلستون: 38
کتاب باز: 1
یونی تهران: 38

Confirmed programs found: 9 / 9
Missing confirmed programs: NONE

UNMAPPED PRIMARY FILES
--------------------------------------------------------------------------------
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/اقیانوس آرام 1.md
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/اقیانوس آرام 2.md
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/بانک ملت 1.md
 - data/transcripts/final/chehelstion_finup_ketabba

<div style="direction: rtl; white-space: normal; line-height: 1;">

### تکمیل Human-confirmed Program Map برای Sourceهای باقی‌مانده

در مرحله قبل Scope فایل‌های Transcript پاک‌سازی شد و تمام ۹ برنامه اولیه روی Disk شناسایی شدند.

همچنین `15` فایل Primary با وضعیت:

`UNMAPPED_PROGRAM`

باقی ماندند.

بر اساس اطلاعات انسانی تأییدشده، این فایل‌ها متعلق به چهار مجموعه مستقل هستند:

- `بانک ملت`
- `اقیانوس آرام`
- `شراکت`
- `میزان`

اطلاعات تأییدشده درباره این مجموعه‌ها:

#### بانک ملت

- اجرای تک‌نفره است.
- گوینده دکتر کمیل رودی است.
- Speaker Attribution برای این مجموعه ضروری نیست.

#### اقیانوس آرام

- یک برنامه تلویزیونی است.
- دکتر کمیل رودی در آن حضور دارد.
- ساختار کامل سایر Speakerها هنوز مشخص نیست.
- بنابراین Speaker Attribution باید در مراحل بعد حفظ و بررسی شود و هیچ Speaker ناشناخته‌ای حدس زده نمی‌شود.

#### شراکت

- محتوای اصلی به‌صورت توضیح تک‌نفره توسط دکتر کمیل رودی است.
- ممکن است در بخش‌هایی صدای فرد دیگری برای تأیید یا تعامل کوتاه وجود داشته باشد.
- بنابراین دکتر رودی Primary Speaker است، اما وجود Secondary Speaker احتمالی باید حفظ شود.

#### میزان

- یک دوره آموزشی است.
- دکتر کمیل رودی در آن حضور دارد.
- تعداد دقیق Speakerها هنوز Human-confirmed نیست.
- بنابراین Speaker Structure فعلاً `UNRESOLVED` باقی می‌ماند و بعداً از خود Transcript بررسی خواهد شد.

در نتیجه تعداد Programهای Human-confirmed از `9` به `13` افزایش می‌یابد.

پوشه مشترک:

`chehelstion_finup_ketabbaz_bankmellat`

نیازی به جابه‌جایی فیزیکی فایل‌ها ندارد.

در Inventory هر فایل بر اساس Filename به Program مستقل خودش متصل می‌شود و Physical Path اصلی بدون تغییر حفظ خواهد شد.

در این مرحله هنوز:

- Transcript Ingestion انجام نمی‌شود.
- Speaker Diarization انجام نمی‌شود.
- متن فایل‌ها تغییر نمی‌کند.
- فایل‌ها Move یا Rename نمی‌شوند.

**معیار PASS:**

- هر `13` Program تأییدشده حداقل یک فایل Primary روی Disk داشته باشد.
- هر `183` Primary Transcript Candidate به یک Program مشخص متصل شود.
- تعداد `UNMAPPED_PROGRAM` به صفر برسد.
- Review / Working Material همچنان جدا از Primary Source باقی بماند.

</div>

In [3]:
# Code Cell 03

human_confirmed_program_context.update({
    "بانک ملت": {
        "format": "SOLO",
        "primary_speaker": "Dr. Komeil Roudi",
        "speaker_attribution_required": False,
        "speaker_structure_status": "HUMAN_CONFIRMED",
    },
    "اقیانوس آرام": {
        "format": "TV_PROGRAM",
        "primary_speaker": "Dr. Komeil Roudi",
        "speaker_attribution_required": True,
        "speaker_structure_status": "UNRESOLVED",
    },
    "شراکت": {
        "format": "PRIMARY_SOLO_WITH_POSSIBLE_SECONDARY",
        "primary_speaker": "Dr. Komeil Roudi",
        "speaker_attribution_required": True,
        "speaker_structure_status": "PARTIALLY_CONFIRMED",
    },
    "میزان": {
        "format": "COURSE",
        "primary_speaker": "Dr. Komeil Roudi",
        "speaker_attribution_required": True,
        "speaker_structure_status": "UNRESOLVED",
    },
})

def classify_shared_directory_file_v2(filename):
    stem = Path(filename).stem.strip()

    if stem.startswith("چهلستون"):
        return "چهلستون"

    if stem == "فناپ":
        return "فناپ"

    if stem == "کتاب باز":
        return "کتاب باز"

    if stem.startswith("بانک ملت"):
        return "بانک ملت"

    if stem.startswith("اقیانوس آرام"):
        return "اقیانوس آرام"

    if stem == "شراکت":
        return "شراکت"

    if stem == "میزان":
        return "میزان"

    return "UNMAPPED_PROGRAM"

resolved_primary_transcript_candidates = []

for record in primary_transcript_candidates:
    updated_record = dict(record)

    if record["top_directory"] == shared_directory_name:
        updated_record["canonical_program"] = (
            classify_shared_directory_file_v2(
                record["filename"]
            )
        )

    resolved_primary_transcript_candidates.append(
        updated_record
    )

resolved_program_counts = Counter(
    record["canonical_program"]
    for record in resolved_primary_transcript_candidates
    if record["canonical_program"] != "UNMAPPED_PROGRAM"
)

resolved_unmapped_records = [
    record
    for record in resolved_primary_transcript_candidates
    if record["canonical_program"] == "UNMAPPED_PROGRAM"
]

confirmed_programs_on_disk_v2 = sorted(
    program
    for program in human_confirmed_program_context
    if resolved_program_counts.get(program, 0) > 0
)

missing_confirmed_programs_v2 = sorted(
    set(human_confirmed_program_context)
    - set(confirmed_programs_on_disk_v2)
)

mapped_primary_count_v2 = sum(
    1
    for record in resolved_primary_transcript_candidates
    if record["canonical_program"] != "UNMAPPED_PROGRAM"
)

program_inventory_gate_pass = (
    len(human_confirmed_program_context) == 13
    and len(resolved_primary_transcript_candidates) == 183
    and mapped_primary_count_v2 == 183
    and not resolved_unmapped_records
    and not missing_confirmed_programs_v2
    and len(review_materials) == 30
)

print("TRANSCRIPT INVENTORY — HUMAN-CONFIRMED PROGRAM MAP")
print("-" * 80)
print(
    "Human-confirmed programs:",
    len(human_confirmed_program_context),
)
print(
    "Primary transcript candidates:",
    len(resolved_primary_transcript_candidates),
)
print(
    "Mapped primary files:",
    mapped_primary_count_v2,
)
print(
    "Unmapped primary files:",
    len(resolved_unmapped_records),
)
print(
    "Review / working materials:",
    len(review_materials),
)

print("\nPROGRAM COVERAGE")
print("-" * 80)

for program in human_confirmed_program_context:
    context = human_confirmed_program_context[program]

    print(
        f"{program}: "
        f"{resolved_program_counts.get(program, 0)} files "
        f"| format={context['format']} "
        f"| attribution_required="
        f"{context['speaker_attribution_required']}"
    )

print()
print(
    "Confirmed programs found:",
    len(confirmed_programs_on_disk_v2),
    "/",
    len(human_confirmed_program_context),
)

print(
    "Missing confirmed programs:",
    missing_confirmed_programs_v2
    if missing_confirmed_programs_v2
    else "NONE",
)

print(
    "Remaining unmapped files:",
    len(resolved_unmapped_records),
)

if resolved_unmapped_records:
    for record in resolved_unmapped_records:
        print(
            " -",
            record["relative_path"],
        )

print()
print(
    "TRANSCRIPT HUMAN-CONFIRMED PROGRAM MAP GATE:",
    "PASS" if program_inventory_gate_pass else "FAIL",
)

TRANSCRIPT INVENTORY — HUMAN-CONFIRMED PROGRAM MAP
--------------------------------------------------------------------------------
Human-confirmed programs: 13
Primary transcript candidates: 183
Mapped primary files: 183
Unmapped primary files: 0
Review / working materials: 30

PROGRAM COVERAGE
--------------------------------------------------------------------------------
در جستجوی خوشبختی: 29 files | format=SOLO | attribution_required=False
تربیت مربی: 14 files | format=SOLO | attribution_required=False
ام کیو: 8 files | format=SOLO | attribution_required=False
فناپ: 1 files | format=SOLO | attribution_required=False
عصر شیرین: 26 files | format=MULTI_SPEAKER | attribution_required=True
خانه تو: 13 files | format=TWO_SPEAKER | attribution_required=True
چهلستون: 38 files | format=MULTI_SPEAKER_RADIO | attribution_required=True
کتاب باز: 1 files | format=TWO_SPEAKER | attribution_required=True
یونی تهران: 38 files | format=TWO_SPEAKER | attribution_required=True
بانک ملت: 11 files | 

<div style="direction: rtl; white-space: normal; line-height: 1;">

### Fingerprint و ممیزی شماره‌گذاری فایل‌های Transcript

پس از تکمیل Human-confirmed Program Map، هر `183` فایل Primary Transcript Candidate به یک Program مشخص متصل شده است.

در این مرحله Inventory از سطح نام Program به سطح خود فایل‌ها می‌رود.

برای هر فایل موارد زیر ثبت می‌شود:

- مسیر اصلی فایل
- Program
- نام فایل
- اندازه فایل
- `SHA-256`
- تعداد Byte
- تعداد Character
- تعداد Line
- شماره Episode در صورت قابل‌استخراج بودن از Filename

هدف این مرحله هنوز تحلیل محتوای Transcript نیست.

همچنین برای Programهایی که Filenameهای شماره‌دار دارند، Sequence بررسی می‌شود تا موارد زیر مشخص شوند:

- شماره‌های موجود
- شماره‌های گمشده بین Minimum و Maximum
- شماره‌های تکراری
- فایل‌هایی که شماره Episode آن‌ها از Filename قابل‌تشخیص نیست

Hashها نیز برای تشخیص `Exact Duplicate` استفاده می‌شوند.

وجود Gap الزاماً به معنی FAIL نیست؛ چون ممکن است Source واقعاً ناقص باشد یا فایل جایگزین در Review Material وجود داشته باشد.

در این مرحله هیچ فایل Review به Primary Source ارتقا داده نمی‌شود.

به‌خصوص اگر برای یک Episode گمشده Candidate دیگری در پوشه‌های Review وجود داشته باشد، فقط به‌عنوان Evidence برای بررسی بعدی ثبت خواهد شد و بدون Validation جایگزین نمی‌شود.

**معیار PASS:**

- هر `183` فایل Primary بدون Error Fingerprint شود.
- Hash تمام فایل‌ها ساخته شود.
- Sequence هر Program شماره‌دار بدون حدس گزارش شود.
- Duplicateهای دقیق در صورت وجود مشخص شوند.
- هیچ فایل Primary در این مرحله تغییر، Rename یا Move نشود.

</div>

In [4]:
# Code Cell 04

import hashlib
import re
from collections import Counter, defaultdict

def extract_episode_number(filename):
    stem = Path(filename).stem.strip()

    patterns = [
        r"^episode[-_\s]*(\d+)$",
        r"^(\d+)$",
        r"^.+?\s+(\d+)$",
    ]

    for pattern in patterns:
        match = re.match(
            pattern,
            stem,
            flags=re.IGNORECASE,
        )

        if match:
            return int(match.group(1))

    return None

transcript_file_inventory = []

for record in resolved_primary_transcript_candidates:
    path = project_root / record["relative_path"]

    raw_bytes = path.read_bytes()

    try:
        text = raw_bytes.decode("utf-8")
        decode_status = "UTF8"
    except UnicodeDecodeError:
        text = raw_bytes.decode(
            "utf-8",
            errors="replace",
        )
        decode_status = "UTF8_WITH_REPLACEMENT"

    transcript_file_inventory.append({
        "relative_path": record["relative_path"],
        "filename": record["filename"],
        "canonical_program": record["canonical_program"],
        "suffix": record["suffix"],
        "size_bytes": len(raw_bytes),
        "sha256": hashlib.sha256(
            raw_bytes
        ).hexdigest(),
        "character_count": len(text),
        "line_count": len(text.splitlines()),
        "episode_number": extract_episode_number(
            record["filename"]
        ),
        "decode_status": decode_status,
    })

hash_groups = defaultdict(list)

for record in transcript_file_inventory:
    hash_groups[
        record["sha256"]
    ].append(record)

exact_duplicate_groups = {
    sha256: records
    for sha256, records in hash_groups.items()
    if len(records) > 1
}

sequence_audit = {}

for program in human_confirmed_program_context:
    records = [
        record
        for record in transcript_file_inventory
        if record["canonical_program"] == program
    ]

    numbered_records = [
        record
        for record in records
        if record["episode_number"] is not None
    ]

    unnumbered_records = [
        record
        for record in records
        if record["episode_number"] is None
    ]

    episode_numbers = [
        record["episode_number"]
        for record in numbered_records
    ]

    duplicate_episode_numbers = sorted({
        number
        for number in episode_numbers
        if episode_numbers.count(number) > 1
    })

    if episode_numbers:
        minimum_episode = min(episode_numbers)
        maximum_episode = max(episode_numbers)

        expected_range = set(
            range(
                minimum_episode,
                maximum_episode + 1,
            )
        )

        missing_episode_numbers = sorted(
            expected_range
            - set(episode_numbers)
        )
    else:
        minimum_episode = None
        maximum_episode = None
        missing_episode_numbers = []

    sequence_audit[program] = {
        "file_count": len(records),
        "numbered_file_count": len(numbered_records),
        "unnumbered_file_count": len(unnumbered_records),
        "episode_numbers": sorted(episode_numbers),
        "minimum_episode": minimum_episode,
        "maximum_episode": maximum_episode,
        "missing_episode_numbers": missing_episode_numbers,
        "duplicate_episode_numbers": duplicate_episode_numbers,
        "unnumbered_files": [
            record["filename"]
            for record in unnumbered_records
        ],
    }

review_episode_21_candidates = [
    record["relative_path"]
    for record in review_materials
    if re.search(
        r"(^|[^0-9])21([^0-9]|$)",
        record["filename"],
    )
]

fingerprint_pass = (
    len(transcript_file_inventory) == 183
    and all(
        record["sha256"]
        and len(record["sha256"]) == 64
        for record in transcript_file_inventory
    )
    and all(
        record["size_bytes"] > 0
        for record in transcript_file_inventory
    )
)

print("TRANSCRIPT FILE FINGERPRINT SUMMARY")
print("-" * 80)
print(
    "Primary files fingerprinted:",
    len(transcript_file_inventory),
)
print(
    "Exact duplicate hash groups:",
    len(exact_duplicate_groups),
)
print(
    "UTF-8 decode issues:",
    sum(
        record["decode_status"]
        != "UTF8"
        for record in transcript_file_inventory
    ),
)

print("\nPROGRAM SEQUENCE AUDIT")
print("-" * 80)

for program, audit in sequence_audit.items():
    print(program)
    print(
        "    Files:",
        audit["file_count"],
        "| Numbered:",
        audit["numbered_file_count"],
        "| Unnumbered:",
        audit["unnumbered_file_count"],
    )

    if audit["numbered_file_count"]:
        print(
            "    Range:",
            f"{audit['minimum_episode']}"
            f"–{audit['maximum_episode']}",
        )
        print(
            "    Missing:",
            audit["missing_episode_numbers"]
            if audit["missing_episode_numbers"]
            else "NONE",
        )
        print(
            "    Duplicate episode numbers:",
            audit["duplicate_episode_numbers"]
            if audit["duplicate_episode_numbers"]
            else "NONE",
        )

    if audit["unnumbered_files"]:
        print(
            "    Unnumbered files:",
            audit["unnumbered_files"],
        )

print("\nEXACT DUPLICATE FILE GROUPS")
print("-" * 80)

if exact_duplicate_groups:
    for sha256, records in exact_duplicate_groups.items():
        print("SHA-256:", sha256)

        for record in records:
            print(
                " -",
                record["relative_path"],
            )
else:
    print("NONE")

print("\nREVIEW MATERIAL — EPISODE 21 NAME CANDIDATES")
print("-" * 80)

if review_episode_21_candidates:
    for path in review_episode_21_candidates:
        print(" -", path)
else:
    print("NONE")

print()
print(
    "TRANSCRIPT FILE FINGERPRINT GATE:",
    "PASS" if fingerprint_pass else "FAIL",
)

TRANSCRIPT FILE FINGERPRINT SUMMARY
--------------------------------------------------------------------------------
Primary files fingerprinted: 183
Exact duplicate hash groups: 0
UTF-8 decode issues: 0

PROGRAM SEQUENCE AUDIT
--------------------------------------------------------------------------------
در جستجوی خوشبختی
    Files: 29 | Numbered: 29 | Unnumbered: 0
    Range: 1–30
    Missing: [21]
    Duplicate episode numbers: NONE
تربیت مربی
    Files: 14 | Numbered: 14 | Unnumbered: 0
    Range: 1–14
    Missing: NONE
    Duplicate episode numbers: NONE
ام کیو
    Files: 8 | Numbered: 8 | Unnumbered: 0
    Range: 1–8
    Missing: NONE
    Duplicate episode numbers: NONE
فناپ
    Files: 1 | Numbered: 0 | Unnumbered: 1
    Unnumbered files: ['فناپ.md']
عصر شیرین
    Files: 26 | Numbered: 26 | Unnumbered: 0
    Range: 1–26
    Missing: NONE
    Duplicate episode numbers: NONE
خانه تو
    Files: 13 | Numbered: 13 | Unnumbered: 0
    Range: 1–13
    Missing: NONE
    Duplicate episo

<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی شواهد Episode 21 در مجموعه «در جستجوی خوشبختی»

Fingerprint تمام `183` فایل Primary با موفقیت انجام شد:

`TRANSCRIPT FILE FINGERPRINT GATE: PASS`

نتیجه Sequence Audit نشان داد تنها Gap عددی در کل Inventory مربوط به:

`در جستجوی خوشبختی — Episode 21`

است.

هم‌زمان یک Candidate مشخص در Review Material پیدا شد:

`extracted/missing-core/final-c-21_from_site-32.md`

نام مسیر نشان می‌دهد این فایل ممکن است برای پوشش Episode گمشده ایجاد شده باشد، اما نام فایل و محل قرارگیری آن به‌تنهایی برای پذیرش کافی نیست.

در این مرحله شواهد مربوط به این Candidate بررسی می‌شود:

- خود فایل Candidate قسمت 21
- Episode 20 و 22 از Primary Source
- فایل `README.md` مربوط به Review Material
- فایل `CURATED_REVIEW.md`
- Hash، اندازه، تعداد خطوط و متن ابتدایی فایل‌ها
- اشاره‌های صریح به Episode 21، `final-c-21` یا `site-32`

هدف این مرحله تصمیم‌گیری مبتنی بر Evidence است.

هیچ فایل:

- Move
- Rename
- Delete
- یا به Primary Source اضافه

نمی‌شود.

**معیار PASS:**

تمام Evidenceهای موجود بدون Error خوانده شوند و وضعیت Candidate قسمت 21 برای تصمیم مرحله بعد به‌صورت صریح قابل بررسی باشد.

</div>

In [5]:
# Code Cell 05

import hashlib
import re

jostojo_root = (
    transcript_final_root
    / "dar-jostojo-final-C"
)

episode_20_path = jostojo_root / "20.md"
episode_22_path = jostojo_root / "22.md"

episode_21_candidate_path = (
    jostojo_root
    / "extracted"
    / "missing-core"
    / "final-c-21_from_site-32.md"
)

review_readme_path = (
    jostojo_root
    / "extracted"
    / "README.md"
)

curated_review_path = (
    jostojo_root
    / "extracted"
    / "CURATED_REVIEW.md"
)

evidence_paths = {
    "PRIMARY_EPISODE_20": episode_20_path,
    "MISSING_EPISODE_21_CANDIDATE": episode_21_candidate_path,
    "PRIMARY_EPISODE_22": episode_22_path,
    "REVIEW_README": review_readme_path,
    "CURATED_REVIEW": curated_review_path,
}

def inspect_text_file(path):
    raw_bytes = path.read_bytes()
    text = raw_bytes.decode("utf-8")

    lines = text.splitlines()

    meaningful_lines = [
        line.strip()
        for line in lines
        if line.strip()
    ]

    markdown_headings = [
        line.strip()
        for line in lines
        if re.match(r"^\s*#{1,6}\s+", line)
    ]

    return {
        "relative_path": str(
            path.relative_to(project_root)
        ),
        "size_bytes": len(raw_bytes),
        "sha256": hashlib.sha256(
            raw_bytes
        ).hexdigest(),
        "character_count": len(text),
        "line_count": len(lines),
        "meaningful_line_count": len(meaningful_lines),
        "markdown_headings": markdown_headings,
        "first_meaningful_lines": meaningful_lines[:20],
        "text": text,
    }

episode_21_evidence = {}

missing_evidence_files = []

for label, path in evidence_paths.items():
    if not path.exists():
        missing_evidence_files.append(label)
        continue

    episode_21_evidence[label] = inspect_text_file(
        path
    )

search_patterns = {
    "episode_21": re.compile(
        r"(?<!\d)21(?!\d)",
        re.IGNORECASE,
    ),
    "final_c_21": re.compile(
        r"final[-_ ]?c[-_ ]?21",
        re.IGNORECASE,
    ),
    "site_32": re.compile(
        r"site[-_ ]?32",
        re.IGNORECASE,
    ),
    "missing_core": re.compile(
        r"missing[-_ ]?core",
        re.IGNORECASE,
    ),
}

review_mentions = {}

for label in [
    "REVIEW_README",
    "CURATED_REVIEW",
]:
    if label not in episode_21_evidence:
        continue

    text = episode_21_evidence[label]["text"]

    matched_lines = []

    for line_number, line in enumerate(
        text.splitlines(),
        start=1,
    ):
        matched_patterns = [
            pattern_name
            for pattern_name, pattern
            in search_patterns.items()
            if pattern.search(line)
        ]

        if matched_patterns:
            matched_lines.append({
                "line_number": line_number,
                "patterns": matched_patterns,
                "text": line.strip(),
            })

    review_mentions[label] = matched_lines

episode_21_candidate_hash_unique = False

if "MISSING_EPISODE_21_CANDIDATE" in episode_21_evidence:
    candidate_hash = episode_21_evidence[
        "MISSING_EPISODE_21_CANDIDATE"
    ]["sha256"]

    all_primary_hashes = {
        record["sha256"]
        for record in transcript_file_inventory
    }

    episode_21_candidate_hash_unique = (
        candidate_hash not in all_primary_hashes
    )

candidate_audit_pass = (
    not missing_evidence_files
    and len(episode_21_evidence) == 5
    and episode_21_evidence[
        "MISSING_EPISODE_21_CANDIDATE"
    ]["size_bytes"] > 0
)

print("EPISODE 21 EVIDENCE AUDIT")
print("-" * 80)

for label, record in episode_21_evidence.items():
    print(label)
    print(
        "    Path:",
        record["relative_path"],
    )
    print(
        "    Size:",
        record["size_bytes"],
        "bytes",
    )
    print(
        "    Characters:",
        record["character_count"],
    )
    print(
        "    Lines:",
        record["line_count"],
    )
    print(
        "    SHA-256:",
        record["sha256"],
    )

    if record["markdown_headings"]:
        print(
            "    Headings:",
            record["markdown_headings"][:10],
        )

    print("    First meaningful lines:")

    for line in record["first_meaningful_lines"][:8]:
        print(
            "       ",
            line[:300],
        )

    print()

print("REVIEW EVIDENCE MENTIONS")
print("-" * 80)

for label in [
    "REVIEW_README",
    "CURATED_REVIEW",
]:
    print(label)

    matches = review_mentions.get(label, [])

    if matches:
        for item in matches[:30]:
            print(
                f"    L{item['line_number']}:",
                item["patterns"],
                "→",
                item["text"][:400],
            )
    else:
        print("    NONE")

    print()

print("CANDIDATE STATUS")
print("-" * 80)
print(
    "Missing evidence files:",
    missing_evidence_files
    if missing_evidence_files
    else "NONE",
)
print(
    "Candidate hash unique vs current Primary corpus:",
    episode_21_candidate_hash_unique,
)
print(
    "Candidate currently remains:",
    "REVIEW_OR_WORKING_MATERIAL",
)
print()
print(
    "EPISODE 21 EVIDENCE AUDIT GATE:",
    "PASS" if candidate_audit_pass else "FAIL",
)

EPISODE 21 EVIDENCE AUDIT
--------------------------------------------------------------------------------
PRIMARY_EPISODE_20
    Path: data/transcripts/final/dar-jostojo-final-C/20.md
    Size: 33476 bytes
    Characters: 19456
    Lines: 1317
    SHA-256: c15dffd0e7d0a7c311e3f2a27418f73dec7c9095f1e13a2c7765598cd7434c60
    Headings: ['# در جستجوی خوشبختی — قسمت 20']
    First meaningful lines:
        # در جستجوی خوشبختی — قسمت 20
        **دکتر کمیل رودی**
        سلام
        **دکتر کمیل رودی**
        در موضوع جذاب سرمایه‌گذاری هستیم
        **دکتر کمیل رودی**
        سرمایه‌گذاری اونقدر جذابه که معمولا از حول حلیم میفتیم تو دیگ
        **دکتر کمیل رودی**

MISSING_EPISODE_21_CANDIDATE
    Path: data/transcripts/final/dar-jostojo-final-C/extracted/missing-core/final-c-21_from_site-32.md
    Size: 12827 bytes
    Characters: 7067
    Lines: 51
    SHA-256: def83f3eb91e70982ec21891a090cd66304918dbfe099724580a470ea3711e43
    Headings: ['# Candidate recovery for missing Final-C lesson

<div style="direction: rtl; white-space: normal; line-height: 1;">

### ثبت Episode 21 به‌عنوان Source گمشده

بررسی شواهد مربوط به Episode 21 نشان داد که مجموعه:

`در جستجوی خوشبختی`

در Primary Transcript Folder دارای Episodeهای `1–30` است، اما فایل Episode `21` در اختیار فعلی پروژه نیست.

یک فایل Review با نام:

`final-c-21_from_site-32.md`

وجود دارد که قبلاً به‌عنوان Candidate بازیابی قسمت 21 نگهداری شده است.

اما بر اساس تأیید انسانی، این فایل به‌عنوان Episode 21 واقعی تأیید نشده است.

بنابراین وضعیت رسمی Episode 21 در Inventory:

`MISSING_SOURCE`

ثبت می‌شود.

Candidate موجود نیز حذف نمی‌شود و با وضعیت:

`RECOVERY_CANDIDATE_NOT_ACCEPTED`

برای بررسی احتمالی آینده حفظ خواهد شد.

این تصمیم باعث می‌شود Pipeline کمبود واقعی Source را پنهان نکند.

اگر در آینده فایل واقعی Episode 21 پیدا شود، می‌توان همان Source جدید را با Hash و Provenance مستقل به Inventory اضافه کرد.

در این مرحله:

- هیچ Review Material به Primary Corpus اضافه نمی‌شود.
- هیچ فایل Move یا Rename نمی‌شود.
- Candidate مربوط به Site 32 حذف نمی‌شود.
- Transcript Ingestion هنوز انجام نمی‌شود.

**معیار PASS:**

- Primary Corpus همچنان `183` فایل داشته باشد.
- مجموعه `در جستجوی خوشبختی` دارای `29` فایل واقعی باشد.
- Episode `21` تنها Gap این مجموعه باقی بماند.
- Candidate مربوط به Site 32 به Primary Corpus اضافه نشود.
- هیچ Review Material دیگری Promote نشود.

</div>

In [6]:
# Code Cell 06

logical_transcript_sources = [
    dict(record)
    for record in transcript_file_inventory
]

for record in logical_transcript_sources:
    record["inventory_source_status"] = "PRIMARY_SOURCE"
    record["provenance_basis"] = "PRIMARY_TRANSCRIPT_DIRECTORY"

episode_21_candidate = episode_21_evidence[
    "MISSING_EPISODE_21_CANDIDATE"
]

episode_21_inventory_gap = {
    "canonical_program": "در جستجوی خوشبختی",
    "episode_number": 21,
    "inventory_status": "MISSING_SOURCE",
    "expected_source_role": "PRIMARY_TRANSCRIPT",
    "human_confirmation": (
        "Actual Episode 21 source is currently unavailable."
    ),
    "future_action": (
        "Add and validate the authentic Episode 21 source "
        "if it becomes available later."
    ),
}

episode_21_recovery_candidate = {
    "relative_path": episode_21_candidate["relative_path"],
    "filename": Path(
        episode_21_candidate["relative_path"]
    ).name,
    "canonical_program": "در جستجوی خوشبختی",
    "candidate_episode_number": 21,
    "sha256": episode_21_candidate["sha256"],
    "size_bytes": episode_21_candidate["size_bytes"],
    "inventory_source_status": (
        "RECOVERY_CANDIDATE_NOT_ACCEPTED"
    ),
    "provenance_basis": (
        "CURATED_REVIEW_SITE_32_RECOVERY_CANDIDATE"
    ),
    "physical_location_preserved": True,
    "included_in_primary_corpus": False,
}

logical_jostojo_records = [
    record
    for record in logical_transcript_sources
    if record["canonical_program"]
    == "در جستجوی خوشبختی"
]

logical_jostojo_episode_numbers = sorted(
    record["episode_number"]
    for record in logical_jostojo_records
    if record["episode_number"] is not None
)

expected_jostojo_episodes = set(
    range(1, 31)
)

logical_jostojo_missing = sorted(
    expected_jostojo_episodes
    - set(logical_jostojo_episode_numbers)
)

logical_jostojo_counts = Counter(
    logical_jostojo_episode_numbers
)

logical_jostojo_duplicates = sorted(
    episode
    for episode, count
    in logical_jostojo_counts.items()
    if count > 1
)

review_promotions = [
    record
    for record in logical_transcript_sources
    if "/extracted/" in record["relative_path"]
]

episode_21_missing_decision_pass = (
    len(logical_transcript_sources) == 183
    and len(logical_jostojo_records) == 29
    and logical_jostojo_missing == [21]
    and not logical_jostojo_duplicates
    and episode_21_inventory_gap[
        "inventory_status"
    ] == "MISSING_SOURCE"
    and episode_21_recovery_candidate[
        "inventory_source_status"
    ] == "RECOVERY_CANDIDATE_NOT_ACCEPTED"
    and not episode_21_recovery_candidate[
        "included_in_primary_corpus"
    ]
    and not review_promotions
)

print("EPISODE 21 INVENTORY DECISION")
print("-" * 80)
print(
    "Episode 21 status:",
    episode_21_inventory_gap[
        "inventory_status"
    ],
)
print(
    "Recovery candidate status:",
    episode_21_recovery_candidate[
        "inventory_source_status"
    ],
)
print(
    "Candidate included in Primary corpus:",
    episode_21_recovery_candidate[
        "included_in_primary_corpus"
    ],
)
print(
    "Candidate path:",
    episode_21_recovery_candidate[
        "relative_path"
    ],
)

print("\nPRIMARY CORPUS COVERAGE")
print("-" * 80)
print(
    "Primary transcript records:",
    len(logical_transcript_sources),
)
print(
    "در جستجوی خوشبختی Primary files:",
    len(logical_jostojo_records),
)
print(
    "Expected episode range:",
    "1–30",
)
print(
    "Missing episodes:",
    logical_jostojo_missing
    if logical_jostojo_missing
    else "NONE",
)
print(
    "Duplicate episode numbers:",
    logical_jostojo_duplicates
    if logical_jostojo_duplicates
    else "NONE",
)
print(
    "Review materials promoted:",
    len(review_promotions),
)

print()
print(
    "EPISODE 21 MISSING SOURCE DECISION GATE:",
    "PASS"
    if episode_21_missing_decision_pass
    else "FAIL",
)

EPISODE 21 INVENTORY DECISION
--------------------------------------------------------------------------------
Episode 21 status: MISSING_SOURCE
Recovery candidate status: RECOVERY_CANDIDATE_NOT_ACCEPTED
Candidate included in Primary corpus: False
Candidate path: data/transcripts/final/dar-jostojo-final-C/extracted/missing-core/final-c-21_from_site-32.md

PRIMARY CORPUS COVERAGE
--------------------------------------------------------------------------------
Primary transcript records: 183
در جستجوی خوشبختی Primary files: 29
Expected episode range: 1–30
Missing episodes: [21]
Duplicate episode numbers: NONE
Review materials promoted: 0

EPISODE 21 MISSING SOURCE DECISION GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### ممیزی ساختار Markdown و Speaker Labelهای صریح

Episode 21 با وضعیت صحیح:

`MISSING_SOURCE`

ثبت شد و هیچ Review Material به Primary Corpus اضافه نشد.

در این مرحله ساختار داخلی `183` فایل Primary Transcript بررسی می‌شود.

هدف هنوز Transcript Ingestion یا Speaker Diarization نیست.

فقط نشانه‌هایی که به‌صورت صریح داخل خود فایل Markdown وجود دارند بررسی می‌شوند، مانند:

- Heading اصلی فایل
- Labelهای Bold مستقل مانند:
  `**دکتر کمیل رودی**`
- تعداد Speaker Labelهای صریح در هر فایل
- تعداد Speaker Labelهای یکتا در هر Program
- فایل‌هایی که هیچ Speaker Label صریح ندارند
- فایل‌های Multi-speaker که فقط یک Label صریح در آن‌ها دیده می‌شود

هیچ نام Speaker از روی متن گفتگو حدس زده نمی‌شود.

وجود نداشتن Label نیز الزاماً FAIL نیست؛ ممکن است یک فایل تک‌نفره باشد یا ساختار Transcript متفاوتی داشته باشد.

این Audit برای طراحی صحیح مرحله بعدی Inventory و بعداً Transcript Ingestion استفاده خواهد شد.

**معیار PASS:**

- هر `183` فایل Primary بدون Error بررسی شود.
- هر `13` Program در Audit حضور داشته باشد.
- Heading و Speaker Labelهای صریح بدون تغییر متن ثبت شوند.
- هیچ Speaker ناشناخته‌ای به‌صورت حدسی ایجاد نشود.

</div>

In [7]:
# Code Cell 07

import re
from collections import Counter, defaultdict

bold_only_pattern = re.compile(
    r"^\s*\*\*(.+?)\*\*\s*$"
)

heading_pattern = re.compile(
    r"^\s*#{1,6}\s+(.+?)\s*$"
)

markdown_structure_inventory = []

speaker_labels_by_program = defaultdict(Counter)

for record in logical_transcript_sources:
    path = project_root / record["relative_path"]
    text = path.read_text(encoding="utf-8")

    lines = text.splitlines()

    headings = []
    explicit_speaker_labels = []

    for line in lines:
        heading_match = heading_pattern.match(line)

        if heading_match:
            headings.append(
                heading_match.group(1).strip()
            )

        speaker_match = bold_only_pattern.match(line)

        if speaker_match:
            label = speaker_match.group(1).strip()

            if label:
                explicit_speaker_labels.append(label)

    unique_speaker_labels = list(
        dict.fromkeys(explicit_speaker_labels)
    )

    markdown_structure_inventory.append({
        "relative_path": record["relative_path"],
        "canonical_program": record["canonical_program"],
        "episode_number": record["episode_number"],
        "heading_count": len(headings),
        "headings": headings,
        "explicit_speaker_label_count": len(
            explicit_speaker_labels
        ),
        "unique_explicit_speaker_labels": (
            unique_speaker_labels
        ),
        "unique_explicit_speaker_count": len(
            unique_speaker_labels
        ),
    })

    for label in explicit_speaker_labels:
        speaker_labels_by_program[
            record["canonical_program"]
        ][label] += 1

files_without_explicit_speaker_labels = [
    record
    for record in markdown_structure_inventory
    if record["explicit_speaker_label_count"] == 0
]

multi_speaker_programs = {
    program
    for program, context
    in human_confirmed_program_context.items()
    if context["speaker_attribution_required"]
}

multi_speaker_files_with_less_than_two_labels = [
    record
    for record in markdown_structure_inventory
    if (
        record["canonical_program"]
        in multi_speaker_programs
        and record["unique_explicit_speaker_count"] < 2
    )
]

program_structure_summary = {}

for program in human_confirmed_program_context:
    records = [
        record
        for record in markdown_structure_inventory
        if record["canonical_program"] == program
    ]

    labels = speaker_labels_by_program[program]

    program_structure_summary[program] = {
        "file_count": len(records),
        "files_with_no_explicit_labels": sum(
            record["explicit_speaker_label_count"] == 0
            for record in records
        ),
        "unique_explicit_labels": list(labels.keys()),
        "label_occurrence_counts": dict(labels),
    }

explicit_structure_audit_pass = (
    len(markdown_structure_inventory) == 183
    and len(program_structure_summary) == 13
    and all(
        summary["file_count"] > 0
        for summary in program_structure_summary.values()
    )
)

print("TRANSCRIPT MARKDOWN STRUCTURE AUDIT")
print("-" * 80)
print(
    "Primary files inspected:",
    len(markdown_structure_inventory),
)
print(
    "Files without explicit speaker labels:",
    len(files_without_explicit_speaker_labels),
)
print(
    "Multi-speaker files with fewer than two explicit labels:",
    len(multi_speaker_files_with_less_than_two_labels),
)

print("\nPROGRAM SPEAKER LABEL SUMMARY")
print("-" * 80)

for program, summary in program_structure_summary.items():
    print(program)
    print(
        "    Files:",
        summary["file_count"],
    )
    print(
        "    Files with no explicit labels:",
        summary["files_with_no_explicit_labels"],
    )

    label_counts = summary["label_occurrence_counts"]

    if label_counts:
        print("    Explicit labels:")

        for label, count in sorted(
            label_counts.items(),
            key=lambda item: (-item[1], item[0]),
        )[:30]:
            print(
                f"      - {label!r}: {count}"
            )
    else:
        print("    Explicit labels: NONE")

print("\nFILES WITHOUT EXPLICIT SPEAKER LABELS")
print("-" * 80)

if files_without_explicit_speaker_labels:
    for record in files_without_explicit_speaker_labels[:50]:
        print(
            " -",
            record["relative_path"],
        )

    if len(files_without_explicit_speaker_labels) > 50:
        print(
            " ...",
            len(files_without_explicit_speaker_labels) - 50,
            "more",
        )
else:
    print("NONE")

print(
    "\nMULTI-SPEAKER FILES WITH < 2 EXPLICIT LABELS"
)
print("-" * 80)

if multi_speaker_files_with_less_than_two_labels:
    for record in multi_speaker_files_with_less_than_two_labels[:50]:
        print(
            " -",
            record["relative_path"],
            "| labels=",
            record["unique_explicit_speaker_labels"],
        )

    if len(
        multi_speaker_files_with_less_than_two_labels
    ) > 50:
        print(
            " ...",
            len(
                multi_speaker_files_with_less_than_two_labels
            ) - 50,
            "more",
        )
else:
    print("NONE")

print()
print(
    "TRANSCRIPT EXPLICIT STRUCTURE AUDIT GATE:",
    "PASS"
    if explicit_structure_audit_pass
    else "FAIL",
)

TRANSCRIPT MARKDOWN STRUCTURE AUDIT
--------------------------------------------------------------------------------
Primary files inspected: 183
Files without explicit speaker labels: 77
Multi-speaker files with fewer than two explicit labels: 43

PROGRAM SPEAKER LABEL SUMMARY
--------------------------------------------------------------------------------
در جستجوی خوشبختی
    Files: 29
    Files with no explicit labels: 0
    Explicit labels:
      - 'دکتر کمیل رودی': 7320
تربیت مربی
    Files: 14
    Files with no explicit labels: 14
    Explicit labels: NONE
ام کیو
    Files: 8
    Files with no explicit labels: 8
    Explicit labels: NONE
فناپ
    Files: 1
    Files with no explicit labels: 1
    Explicit labels: NONE
عصر شیرین
    Files: 26
    Files with no explicit labels: 0
    Explicit labels:
      - 'دکتر کمیل رودی': 440
      - 'خانم متولیان': 213
      - 'یکی از شرکت\u200cکنندگان': 73
      - 'شرکت\u200cکننده': 70
      - 'یکی دیگر از شرکت\u200cکنندگان': 19
      - 'شرکت

<div style="direction: rtl; white-space: normal; line-height: 1;">

### پالایش Speaker Labelها و تعیین Attribution Readiness

ممیزی ساختار Markdown با موفقیت انجام شد:

`TRANSCRIPT EXPLICIT STRUCTURE AUDIT GATE: PASS`

اما مشخص شد که هر عبارت Bold الزاماً Speaker Label نیست.

برای مثال در `عصر شیرین` عبارت‌هایی مانند فرمول‌ها یا جملات آموزشی نیز به‌صورت Bold نوشته شده‌اند و نباید به‌عنوان Speaker شناخته شوند.

بنابراین در این مرحله Labelها به دو گروه تفکیک می‌شوند:

- `SPEAKER_LIKE_LABEL`
- `NON_SPEAKER_BOLD_LABEL`

Speaker-like Label فقط زمانی پذیرفته می‌شود که ساختار نام یا نقش گوینده داشته باشد، مانند:

- دکتر کمیل رودی
- مجری
- استاد
- گوینده
- شرکت‌کننده
- مهمان
- نام اشخاصی که به‌صورت صریح Label شده‌اند

این مرحله همچنین برای هر Program یک وضعیت Speaker Attribution Readiness تعیین می‌کند.

وضعیت‌های ممکن:

`SOLO_ATTRIBUTION_NOT_REQUIRED`

برای Programهای Human-confirmed تک‌نفره.

`EXPLICIT_SPEAKER_STRUCTURE_AVAILABLE`

برای Programهای چندنفره‌ای که Labelهای صریح و قابل‌استفاده دارند.

`STRUCTURAL_REVIEW_REQUIRED`

برای Programهای چندنفره‌ای که Speaker Label کافی ندارند یا ساختار آن‌ها هنوز مشخص نیست.

در این مرحله:

- هیچ Alias به هویت واقعی تبدیل نمی‌شود.
- مثلاً `استاد` هنوز خودکار به `Dr. Komeil Roudi` تبدیل نمی‌شود.
- هیچ Speaker ناشناخته‌ای حدس زده نمی‌شود.
- Transcript Ingestion انجام نمی‌شود.

**معیار PASS:**

- تمام `183` فایل بررسی شوند.
- Boldهای غیرSpeaker از Speaker-like Labelها جدا شوند.
- هر `13` Program یک Attribution Readiness Status داشته باشد.
- Programهای تک‌نفره بدون نیاز به Label قابل قبول باشند.
- Programهای چندنفره فاقد ساختار کافی صریحاً برای Review علامت‌گذاری شوند.

</div>

In [8]:
# Code Cell 08

import re
from collections import Counter, defaultdict

speaker_role_pattern = re.compile(
    r"("
    r"دکتر|استاد|مجری|گوینده|"
    r"شرکت[\u200c\s-]*کننده|"
    r"شرکت[\u200c\s-]*کنندگان|"
    r"مهمان|مصاحبه[\u200c\s-]*کننده|"
    r"خانم|آقای"
    r")",
    re.IGNORECASE,
)

known_person_labels = {
    "دکتر کمیل رودی",
    "فرهاد جم",
    "خانم متولیان",
}

def normalize_label_text(value):
    return re.sub(
        r"\s+",
        " ",
        value.strip(),
    )

def is_speaker_like_label(label):
    normalized = normalize_label_text(label)

    if normalized in known_person_labels:
        return True

    if speaker_role_pattern.search(normalized):
        return True

    return False

refined_structure_inventory = []

speaker_like_labels_by_program = defaultdict(Counter)
non_speaker_bold_labels_by_program = defaultdict(Counter)

for record in logical_transcript_sources:
    path = project_root / record["relative_path"]
    text = path.read_text(encoding="utf-8")

    raw_bold_labels = []

    for line in text.splitlines():
        match = bold_only_pattern.match(line)

        if match:
            label = normalize_label_text(
                match.group(1)
            )

            if label:
                raw_bold_labels.append(label)

    speaker_like_labels = [
        label
        for label in raw_bold_labels
        if is_speaker_like_label(label)
    ]

    non_speaker_bold_labels = [
        label
        for label in raw_bold_labels
        if not is_speaker_like_label(label)
    ]

    unique_speaker_like_labels = list(
        dict.fromkeys(speaker_like_labels)
    )

    refined_structure_inventory.append({
        "relative_path": record["relative_path"],
        "canonical_program": record["canonical_program"],
        "episode_number": record["episode_number"],
        "speaker_like_label_count": len(
            speaker_like_labels
        ),
        "unique_speaker_like_labels": (
            unique_speaker_like_labels
        ),
        "unique_speaker_like_count": len(
            unique_speaker_like_labels
        ),
        "non_speaker_bold_label_count": len(
            non_speaker_bold_labels
        ),
    })

    for label in speaker_like_labels:
        speaker_like_labels_by_program[
            record["canonical_program"]
        ][label] += 1

    for label in non_speaker_bold_labels:
        non_speaker_bold_labels_by_program[
            record["canonical_program"]
        ][label] += 1

speaker_attribution_readiness = {}

for program, context in (
    human_confirmed_program_context.items()
):
    program_records = [
        record
        for record in refined_structure_inventory
        if record["canonical_program"] == program
    ]

    files_with_speaker_like_labels = sum(
        record["speaker_like_label_count"] > 0
        for record in program_records
    )

    files_with_two_or_more_speaker_like_labels = sum(
        record["unique_speaker_like_count"] >= 2
        for record in program_records
    )

    unique_labels = sorted(
        speaker_like_labels_by_program[
            program
        ].keys()
    )

    if not context["speaker_attribution_required"]:
        readiness_status = (
            "SOLO_ATTRIBUTION_NOT_REQUIRED"
        )

    elif (
        files_with_speaker_like_labels
        == len(program_records)
        and len(unique_labels) >= 2
    ):
        readiness_status = (
            "EXPLICIT_SPEAKER_STRUCTURE_AVAILABLE"
        )

    else:
        readiness_status = (
            "STRUCTURAL_REVIEW_REQUIRED"
        )

    speaker_attribution_readiness[program] = {
        "program": program,
        "file_count": len(program_records),
        "speaker_attribution_required": (
            context["speaker_attribution_required"]
        ),
        "files_with_speaker_like_labels": (
            files_with_speaker_like_labels
        ),
        "files_with_two_or_more_speaker_like_labels": (
            files_with_two_or_more_speaker_like_labels
        ),
        "unique_speaker_like_labels": unique_labels,
        "readiness_status": readiness_status,
    }

non_speaker_bold_total = sum(
    sum(counter.values())
    for counter
    in non_speaker_bold_labels_by_program.values()
)

programs_requiring_structural_review = [
    program
    for program, record
    in speaker_attribution_readiness.items()
    if record["readiness_status"]
    == "STRUCTURAL_REVIEW_REQUIRED"
]

speaker_readiness_gate_pass = (
    len(refined_structure_inventory) == 183
    and len(speaker_attribution_readiness) == 13
    and all(
        record["file_count"] > 0
        for record
        in speaker_attribution_readiness.values()
    )
)

print("TRANSCRIPT SPEAKER ATTRIBUTION READINESS")
print("-" * 80)
print(
    "Files inspected:",
    len(refined_structure_inventory),
)
print(
    "Non-speaker Bold labels filtered:",
    non_speaker_bold_total,
)
print(
    "Programs requiring structural review:",
    len(programs_requiring_structural_review),
)

print("\nPROGRAM READINESS")
print("-" * 80)

for program, record in (
    speaker_attribution_readiness.items()
):
    print(program)
    print(
        "    Attribution required:",
        record["speaker_attribution_required"],
    )
    print(
        "    Files:",
        record["file_count"],
    )
    print(
        "    Files with speaker-like labels:",
        record["files_with_speaker_like_labels"],
    )
    print(
        "    Files with >=2 speaker-like labels:",
        record[
            "files_with_two_or_more_speaker_like_labels"
        ],
    )
    print(
        "    Speaker-like labels:",
        record["unique_speaker_like_labels"]
        if record["unique_speaker_like_labels"]
        else "NONE",
    )
    print(
        "    Readiness:",
        record["readiness_status"],
    )

print("\nNON-SPEAKER BOLD LABEL SAMPLES")
print("-" * 80)

for program, counter in (
    non_speaker_bold_labels_by_program.items()
):
    if not counter:
        continue

    print(program)

    for label, count in counter.most_common(10):
        print(
            f"    - {label!r}: {count}"
        )

print("\nSTRUCTURAL REVIEW REQUIRED")
print("-" * 80)

if programs_requiring_structural_review:
    for program in programs_requiring_structural_review:
        print(" -", program)
else:
    print("NONE")

print()
print(
    "TRANSCRIPT SPEAKER READINESS GATE:",
    "PASS"
    if speaker_readiness_gate_pass
    else "FAIL",
)

TRANSCRIPT SPEAKER ATTRIBUTION READINESS
--------------------------------------------------------------------------------
Files inspected: 183
Non-speaker Bold labels filtered: 3
Programs requiring structural review: 5

PROGRAM READINESS
--------------------------------------------------------------------------------
در جستجوی خوشبختی
    Attribution required: False
    Files: 29
    Files with speaker-like labels: 29
    Files with >=2 speaker-like labels: 0
    Speaker-like labels: ['دکتر کمیل رودی']
    Readiness: SOLO_ATTRIBUTION_NOT_REQUIRED
تربیت مربی
    Attribution required: False
    Files: 14
    Files with speaker-like labels: 0
    Files with >=2 speaker-like labels: 0
    Speaker-like labels: NONE
    Readiness: SOLO_ATTRIBUTION_NOT_REQUIRED
ام کیو
    Attribution required: False
    Files: 8
    Files with speaker-like labels: 0
    Files with >=2 speaker-like labels: 0
    Speaker-like labels: NONE
    Readiness: SOLO_ATTRIBUTION_NOT_REQUIRED
فناپ
    Attribution require

<div style="direction: rtl; white-space: normal; line-height: 1;">

### ثبت Human-confirmed Speaker Alias و Attribution Rule

Speaker Readiness Audit با موفقیت انجام شد:

`TRANSCRIPT SPEAKER READINESS GATE: PASS`

اکنون یک بخش از Speaker Attribution بر اساس تأیید انسانی قطعی شده است.

در مجموعه:

`یونی تهران`

Label:

`استاد`

به‌صورت Human-confirmed به:

`Dr. Komeil Roudi`

متصل می‌شود.

بنابراین `استاد` یک Speaker ناشناخته نیست و Alias دکتر کمیل رودی محسوب می‌شود.

همچنین برخی عبارت‌های Bold مانند:

- فرمول‌ها
- تعریف‌ها
- توضیح‌های آموزشی

Speaker Label نیستند.

این محتواها نباید به‌عنوان Speaker جدید ساخته شوند.

اگر چنین محتوایی داخل Turn مربوط به استاد قرار داشته باشد، Speaker Context آن از Turn والد به ارث می‌رسد و به دکتر کمیل رودی تعلق خواهد داشت.

قاعده مهم:

`Bold Content != Speaker Label`

و:

`Instructional Bold Content → inherit current speaker context`

این Rule به معنی نسبت‌دادن خودکار تمام Boldهای هر فایل به دکتر رودی نیست؛ Attribution باید بر اساس Context همان Turn انجام شود.

در این مرحله فقط Alias و Attribution Policy ثبت می‌شود و هنوز Transcript Ingestion انجام نمی‌شود.

**معیار PASS:**

- Alias `استاد → Dr. Komeil Roudi` ثبت شود.
- Boldهای غیرSpeaker به‌عنوان Speaker مستقل شناخته نشوند.
- Human-confirmed Rule از Speaker Detection ماشینی جدا نگه داشته شود.
- هیچ Speaker ناشناخته دیگری حدس زده نشود.

</div>

In [9]:
# Code Cell 09

human_confirmed_speaker_aliases = {
    "یونی تهران": {
        "استاد": "Dr. Komeil Roudi",
    },
}

human_confirmed_attribution_rules = {
    "instructional_bold_is_not_speaker": True,
    "instructional_bold_inherits_current_turn_speaker": True,
    "do_not_create_speaker_from_formula_or_definition": True,
    "do_not_infer_unknown_speaker_identity": True,
}

canonical_primary_speaker = "Dr. Komeil Roudi"

speaker_alias_validation = []

for program, aliases in human_confirmed_speaker_aliases.items():
    observed_labels = set(
        speaker_attribution_readiness[
            program
        ]["unique_speaker_like_labels"]
    )

    for alias, canonical_speaker in aliases.items():
        speaker_alias_validation.append({
            "program": program,
            "alias": alias,
            "canonical_speaker": canonical_speaker,
            "alias_observed": alias in observed_labels,
        })

uni_tehran_alias = human_confirmed_speaker_aliases[
    "یونی تهران"
]["استاد"]

non_speaker_bold_samples = []

for program, counter in (
    non_speaker_bold_labels_by_program.items()
):
    for label, count in counter.items():
        non_speaker_bold_samples.append({
            "program": program,
            "label": label,
            "count": count,
            "speaker_status": "NOT_A_SPEAKER_LABEL",
            "attribution_policy": (
                "INHERIT_CURRENT_TURN_SPEAKER"
            ),
        })

speaker_alias_policy_pass = (
    uni_tehran_alias == canonical_primary_speaker
    and all(
        record["alias_observed"]
        for record in speaker_alias_validation
    )
    and human_confirmed_attribution_rules[
        "instructional_bold_is_not_speaker"
    ]
    and human_confirmed_attribution_rules[
        "instructional_bold_inherits_current_turn_speaker"
    ]
    and human_confirmed_attribution_rules[
        "do_not_infer_unknown_speaker_identity"
    ]
)

print("HUMAN-CONFIRMED SPEAKER ATTRIBUTION POLICY")
print("-" * 80)

print("Speaker aliases:")

for record in speaker_alias_validation:
    print(
        f" - {record['program']}: "
        f"{record['alias']!r} "
        f"→ {record['canonical_speaker']} "
        f"| observed={record['alias_observed']}"
    )

print("\nInstructional Bold content policy:")
print(
    " - Speaker label:",
    "NO",
)
print(
    " - Attribution:",
    "INHERIT_CURRENT_TURN_SPEAKER",
)
print(
    " - Unknown speaker inference:",
    "FORBIDDEN",
)

print("\nNON-SPEAKER BOLD RECORDS")
print("-" * 80)

if non_speaker_bold_samples:
    for record in non_speaker_bold_samples:
        print(
            f" - {record['program']} "
            f"| {record['label']!r} "
            f"| count={record['count']} "
            f"| {record['attribution_policy']}"
        )
else:
    print("NONE")

print()
print(
    "HUMAN-CONFIRMED SPEAKER POLICY GATE:",
    "PASS"
    if speaker_alias_policy_pass
    else "FAIL",
)

HUMAN-CONFIRMED SPEAKER ATTRIBUTION POLICY
--------------------------------------------------------------------------------
Speaker aliases:
 - یونی تهران: 'استاد' → Dr. Komeil Roudi | observed=True

Instructional Bold content policy:
 - Speaker label: NO
 - Attribution: INHERIT_CURRENT_TURN_SPEAKER
 - Unknown speaker inference: FORBIDDEN

NON-SPEAKER BOLD RECORDS
--------------------------------------------------------------------------------
 - عصر شیرین | 'دارایی = سرمایه + بدهی' | count=2 | INHERIT_CURRENT_TURN_SPEAKER
 - عصر شیرین | 'درآمد، هزینه، سود، دارایی، بدهی و سرمایه.' | count=1 | INHERIT_CURRENT_TURN_SPEAKER

HUMAN-CONFIRMED SPEAKER POLICY GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### ممیزی ساختار گفت‌وگو در Programهای نیازمند Structural Review

Speaker Attribution Policy با موفقیت ثبت شد:

`HUMAN-CONFIRMED SPEAKER POLICY GATE: PASS`

پنج Program هنوز وضعیت زیر را دارند:

`STRUCTURAL_REVIEW_REQUIRED`

این Programها عبارت‌اند از:

- چهلستون
- کتاب باز
- اقیانوس آرام
- شراکت
- میزان

در این مرحله هدف تشخیص هویت Speaker نیست.

فقط بررسی می‌شود که خود فایل‌های Transcript چه نشانه‌های ساختاری برای تفکیک Turnهای گفت‌وگو دارند.

موارد بررسی:

- Headingهای Markdown
- خط‌های دارای ساختار `Label: text`
- خط‌های دارای ساختار `Label：text`
- Bold Labelهای موجود
- تعداد Paragraphهای متنی
- نمونه خط‌های ابتدایی فایل
- وجود یا نبود نشانه‌های آشکار تغییر گوینده

هیچ Label ناشناخته‌ای به فرد واقعی نسبت داده نمی‌شود.

اگر یک Program ساختار صریح Speaker نداشته باشد، در Inventory با وضعیت:

`SPEAKER_BOUNDARIES_NOT_EXPLICIT`

ثبت خواهد شد.

این وضعیت به معنی Reject شدن Transcript نیست؛ فقط نشان می‌دهد در Transcript Ingestion نباید هویت Speakerها با حدس ساخته شود.

**معیار PASS:**

- تمام فایل‌های پنج Program بدون Error بررسی شوند.
- ساختار موجود بدون تغییر متن گزارش شود.
- هیچ Speaker Identity جدیدی حدس زده نشود.
- برای هر Program مشخص شود آیا Speaker Boundary صریح در Source وجود دارد یا خیر.

</div>

In [10]:
# Code Cell 10

import re
from collections import Counter, defaultdict

structural_review_programs = [
    "چهلستون",
    "کتاب باز",
    "اقیانوس آرام",
    "شراکت",
    "میزان",
]

colon_label_pattern = re.compile(
    r"^\s*([^:\n]{1,80})\s*[:：]\s*(.+?)\s*$"
)

structural_review_inventory = []
colon_labels_by_program = defaultdict(Counter)

for record in logical_transcript_sources:
    program = record["canonical_program"]

    if program not in structural_review_programs:
        continue

    path = project_root / record["relative_path"]
    text = path.read_text(encoding="utf-8")

    lines = text.splitlines()

    meaningful_lines = [
        line.strip()
        for line in lines
        if line.strip()
    ]

    headings = [
        match.group(1).strip()
        for line in lines
        if (
            match := heading_pattern.match(line)
        )
    ]

    bold_labels = [
        normalize_label_text(match.group(1))
        for line in lines
        if (
            match := bold_only_pattern.match(line)
        )
    ]

    colon_label_records = []

    for line_number, line in enumerate(
        lines,
        start=1,
    ):
        match = colon_label_pattern.match(line)

        if not match:
            continue

        label = normalize_label_text(
            match.group(1)
        )

        content = match.group(2).strip()

        if not label or not content:
            continue

        colon_label_records.append({
            "line_number": line_number,
            "label": label,
            "content": content,
        })

        colon_labels_by_program[
            program
        ][label] += 1

    structural_review_inventory.append({
        "relative_path": record["relative_path"],
        "canonical_program": program,
        "episode_number": record["episode_number"],
        "meaningful_line_count": len(
            meaningful_lines
        ),
        "heading_count": len(headings),
        "headings": headings,
        "bold_label_count": len(bold_labels),
        "bold_labels": list(
            dict.fromkeys(bold_labels)
        ),
        "colon_label_count": len(
            colon_label_records
        ),
        "colon_labels": list(
            dict.fromkeys(
                item["label"]
                for item in colon_label_records
            )
        ),
        "first_meaningful_lines": (
            meaningful_lines[:12]
        ),
    })

program_structural_review = {}

for program in structural_review_programs:
    records = [
        record
        for record in structural_review_inventory
        if record["canonical_program"] == program
    ]

    files_with_bold_labels = sum(
        record["bold_label_count"] > 0
        for record in records
    )

    files_with_colon_labels = sum(
        record["colon_label_count"] > 0
        for record in records
    )

    observed_colon_labels = sorted(
        colon_labels_by_program[
            program
        ].keys()
    )

    explicit_boundary_files = sum(
        (
            record["bold_label_count"] > 0
            or record["colon_label_count"] > 0
        )
        for record in records
    )

    if (
        records
        and explicit_boundary_files
        == len(records)
    ):
        boundary_status = (
            "EXPLICIT_BOUNDARY_STRUCTURE_AVAILABLE"
        )
    elif explicit_boundary_files > 0:
        boundary_status = (
            "PARTIAL_BOUNDARY_STRUCTURE"
        )
    else:
        boundary_status = (
            "SPEAKER_BOUNDARIES_NOT_EXPLICIT"
        )

    program_structural_review[program] = {
        "file_count": len(records),
        "files_with_bold_labels": (
            files_with_bold_labels
        ),
        "files_with_colon_labels": (
            files_with_colon_labels
        ),
        "explicit_boundary_files": (
            explicit_boundary_files
        ),
        "observed_colon_labels": (
            observed_colon_labels
        ),
        "boundary_status": boundary_status,
    }

structural_review_gate_pass = (
    len(program_structural_review) == 5
    and all(
        record["file_count"] > 0
        for record
        in program_structural_review.values()
    )
)

print("TRANSCRIPT STRUCTURAL REVIEW")
print("-" * 80)

for program, summary in (
    program_structural_review.items()
):
    print(program)
    print(
        "    Files:",
        summary["file_count"],
    )
    print(
        "    Files with Bold labels:",
        summary["files_with_bold_labels"],
    )
    print(
        "    Files with colon labels:",
        summary["files_with_colon_labels"],
    )
    print(
        "    Files with any explicit boundary structure:",
        summary["explicit_boundary_files"],
    )
    print(
        "    Colon labels:",
        summary["observed_colon_labels"]
        if summary["observed_colon_labels"]
        else "NONE",
    )
    print(
        "    Boundary status:",
        summary["boundary_status"],
    )

print("\nFILE SAMPLES")
print("-" * 80)

for record in structural_review_inventory:
    print(
        record["relative_path"]
    )
    print(
        "    Bold labels:",
        record["bold_labels"]
        if record["bold_labels"]
        else "NONE",
    )
    print(
        "    Colon labels:",
        record["colon_labels"]
        if record["colon_labels"]
        else "NONE",
    )
    print("    First meaningful lines:")

    for line in record["first_meaningful_lines"]:
        print(
            "       ",
            line[:300],
        )

    print()

print(
    "TRANSCRIPT STRUCTURAL REVIEW GATE:",
    "PASS"
    if structural_review_gate_pass
    else "FAIL",
)

TRANSCRIPT STRUCTURAL REVIEW
--------------------------------------------------------------------------------
چهلستون
    Files: 38
    Files with Bold labels: 0
    Files with colon labels: 38
    Files with any explicit boundary structure: 38
    Colon labels: ['- **برنامه', '- **توضیحات', '- **عنوان/موضوع', '- **قسمت', '- **مدت\u200cزمان کل', '- **منبع اصلی', '- **وضعیت متن', '> بازبین', '> تاریخ بازبینی', '> نوع بازبینی', 'آزادی مالی یک معنای فراتر هم دارد', 'آشنایی کودکان با مفاهیم مالی گاهی نگرانی\u200cهایی در والدین ایجاد می\u200cکند', 'آقای تهمتن', 'آقای داودی', 'آقای دکتر رودی', 'آقای دکتر، در بخش قبلی دو واژه استفاده کردید', 'آقای دکتر، من این\u200cطور متوجه شدم', 'ابزار آموزش هم متنوع بوده', 'از صحبت خانم کاظمی یک نکته مهم گرفتم', 'از طرف دیگر خود تصمیم پس\u200cانداز پیچیده است', 'از طرف دیگر یک نگرش آسیب\u200cزا هم وجود دارد', 'از قبل فکر کنیم', 'از همین\u200cجا یک نکته مهم روشن می\u200cشود', 'از چند نفر پرسیده می\u200cشود', 'از چند کودک پرسیده می\u200cشود', 'از یک کودک پرس

<div style="direction: rtl; white-space: normal; line-height: 1;">

### پالایش Speaker Boundaryهای واقعی در Programهای Structural Review

ممیزی Structural Review با موفقیت انجام شد:

`TRANSCRIPT STRUCTURAL REVIEW GATE: PASS`

اما Audit قبلی نشان داد که Pattern عمومی `Label: text` برای تشخیص Speaker Boundary بیش از حد گسترده است.

برای مثال موارد زیر Speaker نیستند:

- Metadata مانند `برنامه:` و `قسمت:`
- توضیحات بازبینی
- جمله‌های محتوایی مانند `سؤال من این است:`
- عبارت‌های آموزشی که به‌صورت طبیعی دارای `:` هستند

بنابراین در این مرحله فقط Labelهایی Speaker-like محسوب می‌شوند که:

- کوتاه باشند.
- شبیه نام شخص یا نقش گوینده باشند.
- در Metadata Header قرار نداشته باشند.
- از واژه‌هایی مانند `مجری`، `دکتر`، `استاد`، `مهمان`، `گوینده` یا نام شخص تشکیل شده باشند.
- یا نقش گفت‌وگویی صریح مانند `یکی از حاضران` داشته باشند.

برای Programهایی که چنین Boundary صریحی ندارند، Source رد نمی‌شود.

وضعیت آن‌ها فقط به‌صورت:

`SPEAKER_BOUNDARIES_NOT_EXPLICIT`

ثبت خواهد شد تا در Transcript Ingestion از نسبت‌دادن حدسی Speaker جلوگیری شود.

این مرحله هنوز:

- Speaker Identity جدید تولید نمی‌کند.
- Diarization انجام نمی‌دهد.
- Transcript را تغییر نمی‌دهد.
- Ingestion را شروع نمی‌کند.

**معیار PASS:**

- هر پنج Program دوباره با معیار سخت‌گیرانه بررسی شوند.
- Metadata و جمله‌های عادی Speaker Label محسوب نشوند.
- Speaker-like Boundaryهای واقعی در صورت وجود ثبت شوند.
- برای هر Program وضعیت نهایی Boundary Readiness مشخص شود.

</div>

In [11]:
# Code Cell 11

import re
from collections import Counter, defaultdict

strict_speaker_label_pattern = re.compile(
    r"^(?:"
    r"مجری|"
    r"استاد|"
    r"مهمان|"
    r"گوینده(?:\s*[۰-۹0-9]+)?|"
    r"دکتر\s+[\w\u0600-\u06FF\u200c\-]+(?:\s+[\w\u0600-\u06FF\u200c\-]+){0,3}|"
    r"آقای\s+(?:دکتر\s+)?[\w\u0600-\u06FF\u200c\-]+(?:\s+[\w\u0600-\u06FF\u200c\-]+){0,3}|"
    r"خانم\s+(?:دکتر\s+)?[\w\u0600-\u06FF\u200c\-]+(?:\s+[\w\u0600-\u06FF\u200c\-]+){0,3}|"
    r"یکی\s+از\s+حاضران|"
    r"یکی\s+از\s+والدین|"
    r"یکی\s+از\s+شرکت[\u200c\s-]*کنندگان"
    r")$",
    re.IGNORECASE,
)

metadata_prefix_pattern = re.compile(
    r"^\s*(?:[-*>]\s*)?\*\*"
)

def normalize_candidate_label(value):
    value = value.strip()
    value = re.sub(r"^\s*[-*>]\s*", "", value)
    value = value.replace("**", "")
    value = re.sub(r"\s+", " ", value)
    return value.strip()

strict_boundary_inventory = []
strict_labels_by_program = defaultdict(Counter)

for record in logical_transcript_sources:
    program = record["canonical_program"]

    if program not in structural_review_programs:
        continue

    path = project_root / record["relative_path"]
    lines = path.read_text(
        encoding="utf-8"
    ).splitlines()

    in_body = False
    strict_labels = []

    for line in lines:
        stripped = line.strip()

        if stripped.startswith("# متن"):
            in_body = True
            continue

        if not in_body:
            continue

        match = colon_label_pattern.match(line)

        if not match:
            continue

        raw_label = match.group(1).strip()

        if metadata_prefix_pattern.match(raw_label):
            continue

        label = normalize_candidate_label(
            raw_label
        )

        if not label:
            continue

        if len(label) > 60:
            continue

        if strict_speaker_label_pattern.fullmatch(
            label
        ):
            strict_labels.append(label)
            strict_labels_by_program[
                program
            ][label] += 1

    strict_boundary_inventory.append({
        "relative_path": record["relative_path"],
        "canonical_program": program,
        "episode_number": record["episode_number"],
        "strict_speaker_labels": list(
            dict.fromkeys(strict_labels)
        ),
        "strict_speaker_label_count": len(
            strict_labels
        ),
    })

strict_program_boundary_summary = {}

for program in structural_review_programs:
    records = [
        record
        for record in strict_boundary_inventory
        if record["canonical_program"] == program
    ]

    files_with_strict_boundaries = sum(
        record["strict_speaker_label_count"] > 0
        for record in records
    )

    observed_labels = sorted(
        strict_labels_by_program[
            program
        ].keys()
    )

    if (
        records
        and files_with_strict_boundaries
        == len(records)
    ):
        boundary_status = (
            "EXPLICIT_SPEAKER_BOUNDARIES_AVAILABLE"
        )

    elif files_with_strict_boundaries > 0:
        boundary_status = (
            "PARTIAL_EXPLICIT_SPEAKER_BOUNDARIES"
        )

    else:
        boundary_status = (
            "SPEAKER_BOUNDARIES_NOT_EXPLICIT"
        )

    strict_program_boundary_summary[
        program
    ] = {
        "file_count": len(records),
        "files_with_strict_boundaries": (
            files_with_strict_boundaries
        ),
        "observed_speaker_like_labels": (
            observed_labels
        ),
        "boundary_status": boundary_status,
    }

strict_boundary_gate_pass = (
    len(strict_boundary_inventory)
    == sum(
        record["file_count"]
        for record
        in program_structural_review.values()
    )
    and len(
        strict_program_boundary_summary
    ) == 5
)

print("STRICT SPEAKER BOUNDARY AUDIT")
print("-" * 80)

for program, summary in (
    strict_program_boundary_summary.items()
):
    print(program)
    print(
        "    Files:",
        summary["file_count"],
    )
    print(
        "    Files with strict speaker boundaries:",
        summary["files_with_strict_boundaries"],
    )
    print(
        "    Speaker-like labels:",
        summary[
            "observed_speaker_like_labels"
        ]
        if summary[
            "observed_speaker_like_labels"
        ]
        else "NONE",
    )
    print(
        "    Boundary status:",
        summary["boundary_status"],
    )

print("\nFILE-LEVEL STRICT BOUNDARY SAMPLES")
print("-" * 80)

for record in strict_boundary_inventory:
    if not record["strict_speaker_labels"]:
        continue

    print(
        record["relative_path"]
    )
    print(
        "    Labels:",
        record["strict_speaker_labels"],
    )

print()
print(
    "STRICT SPEAKER BOUNDARY AUDIT GATE:",
    "PASS"
    if strict_boundary_gate_pass
    else "FAIL",
)

STRICT SPEAKER BOUNDARY AUDIT
--------------------------------------------------------------------------------
چهلستون
    Files: 38
    Files with strict speaker boundaries: 35
    Speaker-like labels: ['آقای تهمتن', 'آقای داودی', 'آقای دکتر رودی', 'خانم تختی\u200cپور', 'خانم دکتر جلالی', 'خانم کاظمی', 'دکتر اخوان', 'دکتر افراسیابی', 'دکتر افضلی', 'دکتر بهزادی\u200cفرد', 'دکتر حسینی', 'دکتر خیامی', 'دکتر راغفر', 'دکتر رسولی\u200cزاده', 'دکتر رودی', 'دکتر سجادی\u200cنیری', 'دکتر سیمیاری', 'دکتر صدرقاضی', 'دکتر علوی', 'دکتر قلیچ', 'دکتر مسعودیان', 'دکتر ناجی', 'دکتر نبی', 'دکتر نصیری', 'دکتر نواب\u200cپور', 'مجری']
    Boundary status: PARTIAL_EXPLICIT_SPEAKER_BOUNDARIES
کتاب باز
    Files: 1
    Files with strict speaker boundaries: 0
    Speaker-like labels: NONE
    Boundary status: SPEAKER_BOUNDARIES_NOT_EXPLICIT
اقیانوس آرام
    Files: 2
    Files with strict speaker boundaries: 0
    Speaker-like labels: NONE
    Boundary status: SPEAKER_BOUNDARIES_NOT_EXPLICIT
شراکت
    Files: 1


<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی هدفمند فایل‌های بدون Speaker Boundary صریح

Strict Speaker Boundary Audit با موفقیت انجام شد:

`STRICT SPEAKER BOUNDARY AUDIT GATE: PASS`

نتیجه نشان داد:

- `چهلستون` در `35` فایل از `38` فایل دارای Speaker Boundary صریح است.
- سه فایل چهلستون هنوز Boundary صریح قابل‌تشخیص ندارند.
- `کتاب باز`، `اقیانوس آرام`، `شراکت` و `میزان` نیز Speaker Boundary صریح ندارند.

در این مرحله فقط همین فایل‌های unresolved بررسی می‌شوند.

هدف تشخیص این موارد است:

- دقیقاً کدام Episodeهای چهلستون Boundary صریح ندارند.
- آیا در متن آن‌ها نشانه‌هایی مانند `دکتر رودی`، `مجری`، سؤال از مهمان یا تغییر گوینده وجود دارد.
- آیا Metadata فایل اطلاعات Human-usable درباره مهمان یا نوع برنامه دارد.
- آیا Source فقط Speaker Composition را مشخص می‌کند یا واقعاً Turn Boundary نیز دارد.

تفاوت مهم:

`Known participant != Explicit speaker boundary`

برای مثال ممکن است Metadata بگوید مهمان برنامه دکتر کمیل رودی است، اما این به‌تنهایی مشخص نمی‌کند هر جمله توسط چه کسی گفته شده است.

بنابراین در این مرحله هیچ جمله‌ای صرفاً براساس نام مهمان به Speaker خاص نسبت داده نمی‌شود.

همچنین Aliasهای شناخته‌شده زیر در سطح Inventory قابل ثبت هستند:

- `دکتر رودی → Dr. Komeil Roudi`
- `آقای دکتر رودی → Dr. Komeil Roudi`
- `دکتر کمیل رودی → Dr. Komeil Roudi`
- `استاد → Dr. Komeil Roudi` فقط در `یونی تهران`

**معیار PASS:**

- تمام فایل‌های بدون Boundary صریح شناسایی شوند.
- Evidenceهای مرتبط بدون تغییر متن نمایش داده شوند.
- تفاوت Participant Metadata و Speaker Boundary حفظ شود.
- هیچ Speaker Attribution حدسی ایجاد نشود.

</div>

In [12]:
# Code Cell 12

import re

human_confirmed_speaker_aliases.setdefault(
    "چهلستون",
    {},
).update({
    "دکتر رودی": "Dr. Komeil Roudi",
    "آقای دکتر رودی": "Dr. Komeil Roudi",
    "دکتر کمیل رودی": "Dr. Komeil Roudi",
})

human_confirmed_speaker_aliases.setdefault(
    "در جستجوی خوشبختی",
    {},
).update({
    "دکتر کمیل رودی": "Dr. Komeil Roudi",
})

unresolved_boundary_records = [
    record
    for record in strict_boundary_inventory
    if record["strict_speaker_label_count"] == 0
]

target_terms_pattern = re.compile(
    r"("
    r"دکتر|رودی|مجری|مهمان|"
    r"حاضران|والدین|"
    r"می[\u200c\s-]*پرسد|"
    r"می[\u200c\s-]*گوید|"
    r"سؤال|پرسیده"
    r")",
    re.IGNORECASE,
)

unresolved_boundary_evidence = []

for record in unresolved_boundary_records:
    path = project_root / record["relative_path"]
    lines = path.read_text(
        encoding="utf-8"
    ).splitlines()

    metadata_lines = []
    body_evidence_lines = []

    in_body = False

    for line_number, line in enumerate(
        lines,
        start=1,
    ):
        stripped = line.strip()

        if stripped.startswith("# متن"):
            in_body = True
            continue

        if not stripped:
            continue

        if not in_body:
            if stripped.startswith("- **"):
                metadata_lines.append({
                    "line_number": line_number,
                    "text": stripped,
                })

            continue

        if target_terms_pattern.search(stripped):
            body_evidence_lines.append({
                "line_number": line_number,
                "text": stripped,
            })

    unresolved_boundary_evidence.append({
        "relative_path": record["relative_path"],
        "canonical_program": record["canonical_program"],
        "episode_number": record["episode_number"],
        "metadata_lines": metadata_lines,
        "body_evidence_lines": body_evidence_lines,
    })

unresolved_by_program = {}

for program in structural_review_programs:
    records = [
        record
        for record in unresolved_boundary_evidence
        if record["canonical_program"] == program
    ]

    unresolved_by_program[program] = {
        "unresolved_file_count": len(records),
        "files": [
            record["relative_path"]
            for record in records
        ],
    }

expected_unresolved_count = (
    3
    + 1
    + 2
    + 1
    + 1
)

targeted_review_gate_pass = (
    len(unresolved_boundary_records)
    == expected_unresolved_count
    and len(unresolved_boundary_evidence)
    == expected_unresolved_count
    and unresolved_by_program[
        "چهلستون"
    ]["unresolved_file_count"] == 3
    and unresolved_by_program[
        "کتاب باز"
    ]["unresolved_file_count"] == 1
    and unresolved_by_program[
        "اقیانوس آرام"
    ]["unresolved_file_count"] == 2
    and unresolved_by_program[
        "شراکت"
    ]["unresolved_file_count"] == 1
    and unresolved_by_program[
        "میزان"
    ]["unresolved_file_count"] == 1
)

print("TARGETED UNRESOLVED SPEAKER REVIEW")
print("-" * 80)

print(
    "Unresolved files:",
    len(unresolved_boundary_records),
)

print("\nUNRESOLVED FILES BY PROGRAM")
print("-" * 80)

for program, summary in unresolved_by_program.items():
    print(
        program,
        ":",
        summary["unresolved_file_count"],
    )

    for path in summary["files"]:
        print(
            " -",
            path,
        )

print("\nEVIDENCE")
print("-" * 80)

for record in unresolved_boundary_evidence:
    print(
        record["relative_path"]
    )

    print("    Metadata:")

    if record["metadata_lines"]:
        for item in record["metadata_lines"][:15]:
            print(
                f"      L{item['line_number']}:",
                item["text"][:300],
            )
    else:
        print("      NONE")

    print("    Body evidence:")

    if record["body_evidence_lines"]:
        for item in record["body_evidence_lines"][:30]:
            print(
                f"      L{item['line_number']}:",
                item["text"][:400],
            )

        if len(
            record["body_evidence_lines"]
        ) > 30:
            print(
                "      ...",
                len(
                    record["body_evidence_lines"]
                ) - 30,
                "more",
            )
    else:
        print("      NONE")

    print()

print("REGISTERED HUMAN-CONFIRMED ALIASES")
print("-" * 80)

for program in [
    "در جستجوی خوشبختی",
    "چهلستون",
    "یونی تهران",
]:
    aliases = human_confirmed_speaker_aliases.get(
        program,
        {},
    )

    print(program)

    for alias, canonical in aliases.items():
        print(
            f" - {alias!r} → {canonical}"
        )

print()
print(
    "TARGETED UNRESOLVED SPEAKER REVIEW GATE:",
    "PASS"
    if targeted_review_gate_pass
    else "FAIL",
)

TARGETED UNRESOLVED SPEAKER REVIEW
--------------------------------------------------------------------------------
Unresolved files: 8

UNRESOLVED FILES BY PROGRAM
--------------------------------------------------------------------------------
چهلستون : 3
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 1.md
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 2.md
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/چهلستون 38.md
کتاب باز : 1
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/کتاب باز.md
اقیانوس آرام : 2
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/اقیانوس آرام 1.md
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/اقیانوس آرام 2.md
شراکت : 1
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/شراکت.md
میزان : 1
 - data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/میزان.md

EVIDENCE
----------------------------------------------------------------

<div style="direction: rtl; white-space: normal; line-height: 1;">

### تثبیت Speaker Handling Policy برای تمام Programها

Targeted Speaker Review با موفقیت انجام شد:

`TARGETED UNRESOLVED SPEAKER REVIEW GATE: PASS`

اکنون برای هر `13` Program می‌توان یک سیاست رسمی Speaker Handling ثبت کرد.

هدف این Policy تعیین هویت Speakerهای جدید نیست؛ بلکه مشخص می‌کند Source فعلی تا چه سطحی اجازه Attribution معتبر می‌دهد.

سه اصل اصلی:

1. اگر Program با تأیید انسانی کاملاً تک‌نفره باشد، تمام محتوای اصلی آن می‌تواند به Speaker اصلی یعنی:

`Dr. Komeil Roudi`

متصل شود.

2. اگر Source دارای Speaker Label صریح باشد، همان Labelهای Source حفظ می‌شوند و Aliasهای Human-confirmed می‌توانند به هویت Canonical متصل شوند.

3. اگر Speaker Boundary صریح وجود نداشته باشد، حتی اگر شرکت‌کنندگان Program شناخته‌شده باشند، نباید هر جمله با حدس به یک فرد نسبت داده شود.

برای Programهای مختلف وضعیت‌های زیر استفاده می‌شود:

`HUMAN_CONFIRMED_SOLO`

Speaker واحد با تأیید انسانی مشخص است.

`EXPLICIT_MULTI_SPEAKER`

Speaker Labelهای صریح در Source موجود است.

`MIXED_EXPLICIT_AND_CONTEXTUAL`

بخشی از فایل‌ها Label صریح دارند و بخشی فقط Context گفت‌وگویی دارند.

`KNOWN_PARTICIPANTS_UNLABELED_TURNS`

شرکت‌کنندگان شناخته‌شده‌اند اما Turnهای متن Label نشده‌اند.

`PRIMARY_SPEAKER_KNOWN_BOUNDARIES_UNRESOLVED`

Speaker اصلی شناخته‌شده است ولی ساختار کامل Speakerها یا Turnها مشخص نیست.

`NARRATIVE_AUDIENCE_CUES`

Speaker اصلی مشخص است و برای بعضی Turnهای دیگر فقط Roleهای عمومی مانند «یکی از حاضران» وجود دارد.

در تمام موارد:

- Question و Answer Context باید حفظ شود.
- Speaker ناشناخته نباید نام‌گذاری شود.
- Role عمومی مانند `یکی از حاضران` می‌تواند به همان شکل Source-preserved باقی بماند.
- Alias `استاد` در `یونی تهران` به دکتر کمیل رودی متصل است.
- Aliasهای `دکتر رودی` و `آقای دکتر رودی` در `چهلستون` به دکتر کمیل رودی متصل‌اند.
- Boldهای آموزشی Speaker مستقل نیستند و Context Speaker والد را به ارث می‌برند.

این Policy فقط بخشی از Transcript Inventory است و هنوز هیچ Transcript Record یا Segment تولید نمی‌شود.

**معیار PASS:**

- هر `13` Program دقیقاً یک Speaker Handling Policy داشته باشد.
- پنج Program Solo به‌صورت Human-confirmed ثبت شوند.
- Programهای Explicit از Programهای Unresolved جدا شوند.
- هیچ Program با Speaker Boundary نامشخص به‌صورت Full Attribution علامت‌گذاری نشود.
- Unknown Speaker Inference در تمام Policyها ممنوع باشد.

</div>

In [13]:
# Code Cell 13

canonical_primary_speaker = "Dr. Komeil Roudi"

transcript_speaker_policy = {
    "در جستجوی خوشبختی": {
        "policy": "HUMAN_CONFIRMED_SOLO",
        "primary_speaker": canonical_primary_speaker,
        "default_body_attribution_allowed": True,
        "preserve_other_turn_context": False,
    },
    "تربیت مربی": {
        "policy": "HUMAN_CONFIRMED_SOLO",
        "primary_speaker": canonical_primary_speaker,
        "default_body_attribution_allowed": True,
        "preserve_other_turn_context": False,
    },
    "ام کیو": {
        "policy": "HUMAN_CONFIRMED_SOLO",
        "primary_speaker": canonical_primary_speaker,
        "default_body_attribution_allowed": True,
        "preserve_other_turn_context": False,
    },
    "فناپ": {
        "policy": "HUMAN_CONFIRMED_SOLO",
        "primary_speaker": canonical_primary_speaker,
        "default_body_attribution_allowed": True,
        "preserve_other_turn_context": False,
    },
    "بانک ملت": {
        "policy": "HUMAN_CONFIRMED_SOLO",
        "primary_speaker": canonical_primary_speaker,
        "default_body_attribution_allowed": True,
        "preserve_other_turn_context": False,
    },
    "عصر شیرین": {
        "policy": "EXPLICIT_MULTI_SPEAKER",
        "primary_speaker": canonical_primary_speaker,
        "default_body_attribution_allowed": False,
        "preserve_other_turn_context": True,
    },
    "خانه تو": {
        "policy": "EXPLICIT_MULTI_SPEAKER",
        "primary_speaker": canonical_primary_speaker,
        "default_body_attribution_allowed": False,
        "preserve_other_turn_context": True,
    },
    "یونی تهران": {
        "policy": "EXPLICIT_MULTI_SPEAKER",
        "primary_speaker": canonical_primary_speaker,
        "default_body_attribution_allowed": False,
        "preserve_other_turn_context": True,
    },
    "چهلستون": {
        "policy": "MIXED_EXPLICIT_AND_CONTEXTUAL",
        "primary_speaker": canonical_primary_speaker,
        "default_body_attribution_allowed": False,
        "preserve_other_turn_context": True,
        "explicit_boundary_files": 35,
        "contextual_boundary_files": 3,
    },
    "کتاب باز": {
        "policy": "KNOWN_PARTICIPANTS_UNLABELED_TURNS",
        "primary_speaker": canonical_primary_speaker,
        "known_participants": [
            canonical_primary_speaker,
            "سروش صحت",
        ],
        "default_body_attribution_allowed": False,
        "preserve_other_turn_context": True,
    },
    "اقیانوس آرام": {
        "policy": (
            "PRIMARY_SPEAKER_KNOWN_BOUNDARIES_UNRESOLVED"
        ),
        "primary_speaker": canonical_primary_speaker,
        "default_body_attribution_allowed": False,
        "preserve_other_turn_context": True,
    },
    "شراکت": {
        "policy": (
            "PRIMARY_SPEAKER_KNOWN_BOUNDARIES_UNRESOLVED"
        ),
        "primary_speaker": canonical_primary_speaker,
        "default_body_attribution_allowed": False,
        "preserve_other_turn_context": True,
        "possible_secondary_speaker": True,
    },
    "میزان": {
        "policy": "NARRATIVE_AUDIENCE_CUES",
        "primary_speaker": canonical_primary_speaker,
        "default_body_attribution_allowed": False,
        "preserve_other_turn_context": True,
        "generic_roles_observed": [
            "یکی از حاضران",
            "یکی از والدین",
            "یکی می‌پرسد",
        ],
    },
}

for program, policy in transcript_speaker_policy.items():
    policy["unknown_speaker_inference_allowed"] = False
    policy["question_response_context_required"] = (
        policy["preserve_other_turn_context"]
    )

expected_programs = set(
    human_confirmed_program_context
)

policy_programs = set(
    transcript_speaker_policy
)

solo_policy_programs = {
    program
    for program, policy
    in transcript_speaker_policy.items()
    if policy["policy"]
    == "HUMAN_CONFIRMED_SOLO"
}

expected_solo_programs = {
    "در جستجوی خوشبختی",
    "تربیت مربی",
    "ام کیو",
    "فناپ",
    "بانک ملت",
}

unsafe_full_attribution_programs = [
    program
    for program, policy
    in transcript_speaker_policy.items()
    if (
        policy["policy"]
        != "HUMAN_CONFIRMED_SOLO"
        and policy[
            "default_body_attribution_allowed"
        ]
    )
]

unknown_inference_violations = [
    program
    for program, policy
    in transcript_speaker_policy.items()
    if policy[
        "unknown_speaker_inference_allowed"
    ]
]

speaker_policy_gate_pass = (
    len(transcript_speaker_policy) == 13
    and policy_programs == expected_programs
    and solo_policy_programs
    == expected_solo_programs
    and not unsafe_full_attribution_programs
    and not unknown_inference_violations
    and transcript_speaker_policy[
        "چهلستون"
    ]["explicit_boundary_files"] == 35
    and transcript_speaker_policy[
        "چهلستون"
    ]["contextual_boundary_files"] == 3
    and human_confirmed_speaker_aliases[
        "یونی تهران"
    ]["استاد"] == canonical_primary_speaker
)

print("TRANSCRIPT SPEAKER HANDLING POLICY")
print("-" * 80)

for program, policy in (
    transcript_speaker_policy.items()
):
    print(program)
    print(
        "    Policy:",
        policy["policy"],
    )
    print(
        "    Primary speaker:",
        policy["primary_speaker"],
    )
    print(
        "    Default body attribution allowed:",
        policy[
            "default_body_attribution_allowed"
        ],
    )
    print(
        "    Preserve other-turn context:",
        policy[
            "preserve_other_turn_context"
        ],
    )
    print(
        "    Unknown speaker inference:",
        (
            "ALLOWED"
            if policy[
                "unknown_speaker_inference_allowed"
            ]
            else "FORBIDDEN"
        ),
    )

print("\nPOLICY VALIDATION")
print("-" * 80)
print(
    "Programs with policy:",
    len(transcript_speaker_policy),
)
print(
    "Human-confirmed Solo programs:",
    len(solo_policy_programs),
)
print(
    "Unsafe full-attribution programs:",
    unsafe_full_attribution_programs
    if unsafe_full_attribution_programs
    else "NONE",
)
print(
    "Unknown-speaker inference violations:",
    unknown_inference_violations
    if unknown_inference_violations
    else "NONE",
)

print()
print(
    "TRANSCRIPT SPEAKER POLICY GATE:",
    "PASS"
    if speaker_policy_gate_pass
    else "FAIL",
)

TRANSCRIPT SPEAKER HANDLING POLICY
--------------------------------------------------------------------------------
در جستجوی خوشبختی
    Policy: HUMAN_CONFIRMED_SOLO
    Primary speaker: Dr. Komeil Roudi
    Default body attribution allowed: True
    Preserve other-turn context: False
    Unknown speaker inference: FORBIDDEN
تربیت مربی
    Policy: HUMAN_CONFIRMED_SOLO
    Primary speaker: Dr. Komeil Roudi
    Default body attribution allowed: True
    Preserve other-turn context: False
    Unknown speaker inference: FORBIDDEN
ام کیو
    Policy: HUMAN_CONFIRMED_SOLO
    Primary speaker: Dr. Komeil Roudi
    Default body attribution allowed: True
    Preserve other-turn context: False
    Unknown speaker inference: FORBIDDEN
فناپ
    Policy: HUMAN_CONFIRMED_SOLO
    Primary speaker: Dr. Komeil Roudi
    Default body attribution allowed: True
    Preserve other-turn context: False
    Unknown speaker inference: FORBIDDEN
بانک ملت
    Policy: HUMAN_CONFIRMED_SOLO
    Primary speaker: Dr. 

<div style="direction: rtl; white-space: normal; line-height: 1;">

### ممیزی Metadata و Provenance فایل‌های Transcript

Speaker Handling Policy برای تمام `13` Program با موفقیت تثبیت شد:

`TRANSCRIPT SPEAKER POLICY GATE: PASS`

اکنون Inventory از ساختار Speaker به Metadata و Provenance هر Physical File می‌رود.

در این مرحله فقط Header و Metadata فایل‌های Markdown بررسی می‌شود و Body Transcript وارد پردازش محتوایی نمی‌شود.

Metadataهای مورد بررسی شامل مواردی مانند:

- برنامه
- قسمت
- عنوان/موضوع
- توضیحات
- مدت‌زمان
- مدت‌زمان کل
- وضعیت متن
- منبع
- منبع اصلی
- منبع انتشار
- Release
- مهمان
- بازبین
- نوع بازبینی
- تاریخ بازبینی

است.

هدف این مرحله مشخص‌کردن این موارد است:

- چه Metadataهایی واقعاً در فایل‌ها وجود دارد.
- کدام فیلدها در همه یا بخشی از فایل‌ها Missing هستند.
- آیا Program و Episode Metadata با Logical Inventory فعلی سازگار است.
- چه نوع Provenance Evidence برای هر Program در Source موجود است.
- آیا فایل‌ها دارای Source Reference مانند URL، Audio Filename یا Release هستند.

نبودن یک فیلد Metadata به‌تنهایی باعث Reject شدن Transcript نمی‌شود.

در این مرحله هیچ Metadata جدیدی از روی حدس ساخته نمی‌شود.

به‌خصوص:

`Missing metadata != invented metadata`

و:

`Source presence != rights evidence`

بنابراین Source URL یا Audio Filename فقط Provenance محسوب می‌شود و هیچ نتیجه‌ای درباره حقوق استفاده از آن گرفته نمی‌شود.

**معیار PASS:**

- Header هر `183` فایل Primary بدون Error بررسی شود.
- Metadata فقط از خود Source استخراج شود.
- تمام `13` Program در Summary حضور داشته باشند.
- Missing Metadata به‌صورت صریح گزارش شود.
- هیچ مقدار Metadata حدسی ایجاد نشود.
- Episode 21 گمشده همچنان خارج از `183` Primary File باقی بماند.

</div>

In [14]:
# Code Cell 14

import re
from collections import Counter, defaultdict

bullet_metadata_pattern = re.compile(
    r"^\s*-\s*\*\*(.+?)[:：]\*\*\s*(.*?)\s*$"
)

quote_metadata_pattern = re.compile(
    r"^\s*>\s*([^:：]+?)\s*[:：]\s*(.*?)\s*$"
)

def extract_header_metadata(path):
    lines = path.read_text(
        encoding="utf-8"
    ).splitlines()

    metadata = {}
    metadata_records = []

    for line_number, line in enumerate(
        lines,
        start=1,
    ):
        stripped = line.strip()

        if stripped.startswith("# متن"):
            break

        bullet_match = bullet_metadata_pattern.match(
            line
        )

        if bullet_match:
            key = bullet_match.group(1).strip()
            value = bullet_match.group(2).strip()

            metadata.setdefault(
                key,
                [],
            ).append(value)

            metadata_records.append({
                "line_number": line_number,
                "type": "BULLET_METADATA",
                "key": key,
                "value": value,
            })

            continue

        quote_match = quote_metadata_pattern.match(
            line
        )

        if quote_match:
            key = quote_match.group(1).strip()
            value = quote_match.group(2).strip()

            metadata.setdefault(
                key,
                [],
            ).append(value)

            metadata_records.append({
                "line_number": line_number,
                "type": "QUOTE_METADATA",
                "key": key,
                "value": value,
            })

    return {
        "metadata": metadata,
        "metadata_records": metadata_records,
    }

transcript_metadata_inventory = []

for record in logical_transcript_sources:
    path = project_root / record["relative_path"]

    extracted = extract_header_metadata(
        path
    )

    transcript_metadata_inventory.append({
        "relative_path": record["relative_path"],
        "canonical_program": record["canonical_program"],
        "episode_number": record["episode_number"],
        "metadata": extracted["metadata"],
        "metadata_records": extracted[
            "metadata_records"
        ],
    })

metadata_keys_by_program = defaultdict(Counter)

for record in transcript_metadata_inventory:
    program = record["canonical_program"]

    for key in record["metadata"]:
        metadata_keys_by_program[
            program
        ][key] += 1

provenance_key_candidates = {
    "منبع",
    "منبع اصلی",
    "منبع انتشار",
    "Release",
}

program_metadata_summary = {}

for program in human_confirmed_program_context:
    records = [
        record
        for record in transcript_metadata_inventory
        if record["canonical_program"] == program
    ]

    files_with_program_metadata = sum(
        "برنامه" in record["metadata"]
        for record in records
    )

    files_with_episode_metadata = sum(
        "قسمت" in record["metadata"]
        for record in records
    )

    files_with_status_metadata = sum(
        "وضعیت متن" in record["metadata"]
        for record in records
    )

    files_with_provenance_metadata = sum(
        any(
            key in record["metadata"]
            for key in provenance_key_candidates
        )
        for record in records
    )

    all_keys = sorted(
        metadata_keys_by_program[
            program
        ].keys()
    )

    program_metadata_summary[program] = {
        "file_count": len(records),
        "files_with_program_metadata": (
            files_with_program_metadata
        ),
        "files_with_episode_metadata": (
            files_with_episode_metadata
        ),
        "files_with_status_metadata": (
            files_with_status_metadata
        ),
        "files_with_provenance_metadata": (
            files_with_provenance_metadata
        ),
        "observed_metadata_keys": all_keys,
    }

files_without_any_metadata = [
    record
    for record in transcript_metadata_inventory
    if not record["metadata"]
]

files_without_provenance_metadata = [
    record
    for record in transcript_metadata_inventory
    if not any(
        key in record["metadata"]
        for key in provenance_key_candidates
    )
]

program_name_mismatches = []

for record in transcript_metadata_inventory:
    values = record["metadata"].get(
        "برنامه",
        [],
    )

    if not values:
        continue

    observed_program = values[0].strip()

    if normalize_fa_key(
        observed_program
    ) != normalize_fa_key(
        record["canonical_program"]
    ):
        program_name_mismatches.append({
            "relative_path": record["relative_path"],
            "canonical_program": (
                record["canonical_program"]
            ),
            "metadata_program": observed_program,
        })

metadata_audit_gate_pass = (
    len(transcript_metadata_inventory) == 183
    and len(program_metadata_summary) == 13
    and all(
        summary["file_count"] > 0
        for summary
        in program_metadata_summary.values()
    )
)

print("TRANSCRIPT METADATA AND PROVENANCE AUDIT")
print("-" * 80)
print(
    "Primary files inspected:",
    len(transcript_metadata_inventory),
)
print(
    "Files without any parsed metadata:",
    len(files_without_any_metadata),
)
print(
    "Files without provenance metadata:",
    len(files_without_provenance_metadata),
)
print(
    "Program metadata mismatches:",
    len(program_name_mismatches),
)

print("\nPROGRAM METADATA COVERAGE")
print("-" * 80)

for program, summary in (
    program_metadata_summary.items()
):
    print(program)
    print(
        "    Files:",
        summary["file_count"],
    )
    print(
        "    Program metadata:",
        summary["files_with_program_metadata"],
        "/",
        summary["file_count"],
    )
    print(
        "    Episode metadata:",
        summary["files_with_episode_metadata"],
        "/",
        summary["file_count"],
    )
    print(
        "    Text-status metadata:",
        summary["files_with_status_metadata"],
        "/",
        summary["file_count"],
    )
    print(
        "    Provenance metadata:",
        summary[
            "files_with_provenance_metadata"
        ],
        "/",
        summary["file_count"],
    )
    print(
        "    Observed keys:",
        summary["observed_metadata_keys"]
        if summary["observed_metadata_keys"]
        else "NONE",
    )

print("\nFILES WITHOUT ANY PARSED METADATA")
print("-" * 80)

if files_without_any_metadata:
    for record in files_without_any_metadata:
        print(
            " -",
            record["relative_path"],
        )
else:
    print("NONE")

print("\nFILES WITHOUT PROVENANCE METADATA")
print("-" * 80)

if files_without_provenance_metadata:
    for record in files_without_provenance_metadata:
        print(
            " -",
            record["relative_path"],
        )
else:
    print("NONE")

print("\nPROGRAM METADATA MISMATCHES")
print("-" * 80)

if program_name_mismatches:
    for record in program_name_mismatches:
        print(
            " -",
            record["relative_path"],
            "| canonical=",
            record["canonical_program"],
            "| metadata=",
            record["metadata_program"],
        )
else:
    print("NONE")

print()
print(
    "TRANSCRIPT METADATA AUDIT GATE:",
    "PASS"
    if metadata_audit_gate_pass
    else "FAIL",
)

TRANSCRIPT METADATA AND PROVENANCE AUDIT
--------------------------------------------------------------------------------
Primary files inspected: 183
Files without any parsed metadata: 128
Files without provenance metadata: 140
Program metadata mismatches: 1

PROGRAM METADATA COVERAGE
--------------------------------------------------------------------------------
در جستجوی خوشبختی
    Files: 29
    Program metadata: 0 / 29
    Episode metadata: 0 / 29
    Text-status metadata: 0 / 29
    Provenance metadata: 0 / 29
    Observed keys: NONE
تربیت مربی
    Files: 14
    Program metadata: 0 / 14
    Episode metadata: 0 / 14
    Text-status metadata: 0 / 14
    Provenance metadata: 0 / 14
    Observed keys: NONE
ام کیو
    Files: 8
    Program metadata: 0 / 8
    Episode metadata: 0 / 8
    Text-status metadata: 0 / 8
    Provenance metadata: 0 / 8
    Observed keys: NONE
فناپ
    Files: 1
    Program metadata: 1 / 1
    Episode metadata: 1 / 1
    Text-status metadata: 1 / 1
    Provenan

<div style="direction: rtl; white-space: normal; line-height: 1;">

### تثبیت لایه‌های Metadata و Provenance

Metadata Audit با موفقیت انجام شد:

`TRANSCRIPT METADATA AUDIT GATE: PASS`

این Audit نشان داد که همه فایل‌ها Metadata Header یکسان ندارند.

بنابراین برای Inventory نباید نبود Header Metadata را با نبود هویت فایل اشتباه گرفت.

از این مرحله سه لایه Metadata از هم جدا می‌شوند:

`SOURCE_HEADER`

اطلاعاتی که مستقیماً داخل Header فایل نوشته شده‌اند.

`PATH_FILENAME_DERIVED`

اطلاعات ساختاری که بدون تفسیر معنایی از Path و Filename استخراج شده‌اند، مانند Program Mapping و Episode Number.

`HUMAN_CONFIRMED`

اطلاعاتی که توسط انسان تأیید شده‌اند، مانند Program Canonical Name و Speaker Policy.

این لایه‌ها نباید با هم ادغام شوند به‌گونه‌ای که منشأ اطلاعات گم شود.

همچنین Provenance Header می‌تواند از فیلدهای زیر تأمین شود:

- منبع
- منبع اصلی
- منبع انتشار
- منبع آرشیوی
- Release

وجود Source Reference فقط Provenance است و هیچ نتیجه‌ای درباره Rights ایجاد نمی‌کند.

در فایل `فناپ.md`، Metadata Source نام Program را `فیناپ` ثبت کرده است، درحالی‌که Canonical Program مورد استفاده Inventory `فناپ` است.

این اختلاف حذف یا اصلاح نمی‌شود.

به‌جای آن:

`فیناپ`

به‌عنوان:

`SOURCE_NAME_VARIANT`

برای Canonical Program:

`فناپ`

ثبت می‌شود.

**معیار PASS:**

- هر `183` فایل دارای Metadata Provenance Layer مشخص باشد.
- اطلاعات Path-derived از Source Header جدا بماند.
- `منبع آرشیوی` به Provenance Evidence اضافه شود.
- Source Name Variant مربوط به `فیناپ` بدون تغییر فایل ثبت شود.
- هیچ Metadata مفقودی با مقدار حدسی پر نشود.

</div>

In [15]:
# Code Cell 15

expanded_provenance_keys = {
    "منبع",
    "منبع اصلی",
    "منبع انتشار",
    "منبع آرشیوی",
    "Release",
}

program_source_name_variants = {
    "فناپ": {
        "فیناپ": {
            "status": "SOURCE_NAME_VARIANT",
            "canonical_program": "فناپ",
            "basis": "OBSERVED_SOURCE_HEADER",
        },
    },
}

metadata_provenance_layers = []

for file_record in logical_transcript_sources:
    metadata_record = next(
        record
        for record in transcript_metadata_inventory
        if record["relative_path"]
        == file_record["relative_path"]
    )

    source_header_metadata = (
        metadata_record["metadata"]
    )

    source_provenance = {
        key: source_header_metadata[key]
        for key in expanded_provenance_keys
        if key in source_header_metadata
    }

    path_filename_metadata = {
        "canonical_program": (
            file_record["canonical_program"]
        ),
        "episode_number": (
            file_record["episode_number"]
        ),
        "relative_path": (
            file_record["relative_path"]
        ),
        "filename": (
            file_record["filename"]
        ),
    }

    human_confirmed_metadata = {
        "program_confirmed": (
            file_record["canonical_program"]
            in human_confirmed_program_context
        ),
        "speaker_policy": (
            transcript_speaker_policy[
                file_record["canonical_program"]
            ]["policy"]
        ),
    }

    source_program_values = (
        source_header_metadata.get(
            "برنامه",
            [],
        )
    )

    source_program_name = (
        source_program_values[0]
        if source_program_values
        else None
    )

    source_name_variant_status = None

    if source_program_name:
        variant_record = (
            program_source_name_variants
            .get(
                file_record["canonical_program"],
                {},
            )
            .get(source_program_name)
        )

        if variant_record:
            source_name_variant_status = (
                variant_record["status"]
            )

    metadata_provenance_layers.append({
        "relative_path": (
            file_record["relative_path"]
        ),
        "canonical_program": (
            file_record["canonical_program"]
        ),
        "episode_number": (
            file_record["episode_number"]
        ),
        "source_header_metadata": (
            source_header_metadata
        ),
        "source_header_present": bool(
            source_header_metadata
        ),
        "source_provenance": (
            source_provenance
        ),
        "source_provenance_present": bool(
            source_provenance
        ),
        "path_filename_derived": (
            path_filename_metadata
        ),
        "human_confirmed": (
            human_confirmed_metadata
        ),
        "source_program_name": (
            source_program_name
        ),
        "source_name_variant_status": (
            source_name_variant_status
        ),
    })

program_provenance_summary_v2 = {}

for program in human_confirmed_program_context:
    records = [
        record
        for record in metadata_provenance_layers
        if record["canonical_program"] == program
    ]

    program_provenance_summary_v2[program] = {
        "file_count": len(records),
        "source_header_files": sum(
            record["source_header_present"]
            for record in records
        ),
        "source_provenance_files": sum(
            record["source_provenance_present"]
            for record in records
        ),
        "path_filename_derived_files": len(
            records
        ),
        "human_confirmed_files": sum(
            record["human_confirmed"][
                "program_confirmed"
            ]
            for record in records
        ),
    }

files_without_source_provenance_v2 = [
    record
    for record in metadata_provenance_layers
    if not record["source_provenance_present"]
]

source_name_variant_records = [
    record
    for record in metadata_provenance_layers
    if record["source_name_variant_status"]
    == "SOURCE_NAME_VARIANT"
]

layered_metadata_gate_pass = (
    len(metadata_provenance_layers) == 183
    and len(program_provenance_summary_v2) == 13
    and all(
        record["path_filename_derived_files"]
        == record["file_count"]
        for record
        in program_provenance_summary_v2.values()
    )
    and all(
        record["human_confirmed_files"]
        == record["file_count"]
        for record
        in program_provenance_summary_v2.values()
    )
    and len(source_name_variant_records) == 1
    and source_name_variant_records[0][
        "canonical_program"
    ] == "فناپ"
    and source_name_variant_records[0][
        "source_program_name"
    ] == "فیناپ"
)

print("TRANSCRIPT LAYERED METADATA AND PROVENANCE")
print("-" * 80)
print(
    "Files represented:",
    len(metadata_provenance_layers),
)
print(
    "Files without Source Header provenance:",
    len(files_without_source_provenance_v2),
)
print(
    "Source-name variants:",
    len(source_name_variant_records),
)

print("\nPROGRAM PROVENANCE COVERAGE")
print("-" * 80)

for program, summary in (
    program_provenance_summary_v2.items()
):
    print(program)
    print(
        "    Files:",
        summary["file_count"],
    )
    print(
        "    Source Header metadata:",
        summary["source_header_files"],
        "/",
        summary["file_count"],
    )
    print(
        "    Source provenance:",
        summary["source_provenance_files"],
        "/",
        summary["file_count"],
    )
    print(
        "    Path/Filename derived:",
        summary["path_filename_derived_files"],
        "/",
        summary["file_count"],
    )
    print(
        "    Human-confirmed program mapping:",
        summary["human_confirmed_files"],
        "/",
        summary["file_count"],
    )

print("\nSOURCE NAME VARIANTS")
print("-" * 80)

if source_name_variant_records:
    for record in source_name_variant_records:
        print(
            " -",
            record["relative_path"],
            "| source=",
            record["source_program_name"],
            "| canonical=",
            record["canonical_program"],
            "| status=",
            record["source_name_variant_status"],
        )
else:
    print("NONE")

print()
print(
    "LAYERED METADATA AND PROVENANCE GATE:",
    "PASS"
    if layered_metadata_gate_pass
    else "FAIL",
)

TRANSCRIPT LAYERED METADATA AND PROVENANCE
--------------------------------------------------------------------------------
Files represented: 183
Files without Source Header provenance: 140
Source-name variants: 1

PROGRAM PROVENANCE COVERAGE
--------------------------------------------------------------------------------
در جستجوی خوشبختی
    Files: 29
    Source Header metadata: 0 / 29
    Source provenance: 0 / 29
    Path/Filename derived: 29 / 29
    Human-confirmed program mapping: 29 / 29
تربیت مربی
    Files: 14
    Source Header metadata: 0 / 14
    Source provenance: 0 / 14
    Path/Filename derived: 14 / 14
    Human-confirmed program mapping: 14 / 14
ام کیو
    Files: 8
    Source Header metadata: 0 / 8
    Source provenance: 0 / 8
    Path/Filename derived: 8 / 8
    Human-confirmed program mapping: 8 / 8
فناپ
    Files: 1
    Source Header metadata: 1 / 1
    Source provenance: 1 / 1
    Path/Filename derived: 1 / 1
    Human-confirmed program mapping: 1 / 1
عصر شیرین
  

<div style="direction: rtl; white-space: normal; line-height: 1;">

### ساخت Identity Model برای Program، Transcript Unit و Physical File

Layered Metadata و Provenance با موفقیت تثبیت شد:

`LAYERED METADATA AND PROVENANCE GATE: PASS`

اکنون برای Manifest نهایی باید سه مفهوم از هم جدا شوند:

`Program`

مجموعه یا برنامه اصلی، مانند:

- عصر شیرین
- چهلستون
- میزان

`Transcript Unit`

واحد منطقی Transcript داخل یک Program.

در Programهای شماره‌دار این واحد همان Episode شماره‌دار است.

در Programهای تک‌فایلی که شماره Episode از Filename قابل استخراج نیست، یک `Unit ID` ساخته می‌شود؛ اما هیچ Episode Number حدسی ایجاد نمی‌شود.

`Physical File`

فایل واقعی موجود روی Disk.

هویت Physical File با `SHA-256` آن ثبت می‌شود تا نسخه دقیق فایل قابل تشخیص باشد.

بنابراین:

`Episode Identity != Physical File Identity`

اگر در آینده نسخه دیگری از یک Transcript پیدا شود، می‌تواند همان Transcript Unit را داشته باشد اما Physical File متفاوتی با Hash جدید باشد.

برای `عصر شیرین` Season نیز مستقیماً از Path استخراج می‌شود:

- `season-1`
- `season-2`

برای سایر Programها Season حدس زده نمی‌شود.

برای فایل‌هایی مانند `کتاب باز`، Metadata اصلی قسمت نیز در فیلد:

`source_episode_label`

بدون تغییر حفظ می‌شود.

مثلاً:

`فصل ۵ - قسمت ۶۸`

اما این مقدار به Episode Number ساختگی تبدیل نمی‌شود.

Episode 21 گمشده در `در جستجوی خوشبختی` نیز همچنان Physical File ندارد و بعداً به‌صورت Gap Record جداگانه در Manifest ثبت خواهد شد.

**معیار PASS:**

- هر `13` Program یک Program ID یکتا داشته باشد.
- هر `183` Physical File یک Physical File ID یکتا داشته باشد.
- هر فایل دقیقاً به یک Transcript Unit متصل شود.
- Transcript Unit IDها در هر Program یکتا باشند.
- Season فقط از Evidence موجود استخراج شود.
- `عصر شیرین` دقیقاً 13 فایل در Season 1 و 13 فایل در Season 2 داشته باشد.
- هیچ Episode Number حدسی برای فایل‌های Unnumbered ایجاد نشود.

</div>

In [16]:
# Code Cell 16

program_id_map = {
    "در جستجوی خوشبختی": "program:dar-jostojo-ye-khoshbakhti",
    "تربیت مربی": "program:tarbiat-morabi",
    "ام کیو": "program:mq",
    "فناپ": "program:fanap",
    "عصر شیرین": "program:asre-shirin",
    "خانه تو": "program:khane-to",
    "چهلستون": "program:chehelston",
    "کتاب باز": "program:ketab-baz",
    "یونی تهران": "program:uni-tehran",
    "بانک ملت": "program:bank-mellat",
    "اقیانوس آرام": "program:oghyanous-aram",
    "شراکت": "program:sherakat",
    "میزان": "program:mizan",
}

def extract_season_number(relative_path):
    match = re.search(
        r"(?:^|/)season-(\d+)(?:/|$)",
        relative_path,
        flags=re.IGNORECASE,
    )

    if match:
        return int(match.group(1))

    return None

identity_inventory = []

for file_record in logical_transcript_sources:
    program = file_record["canonical_program"]
    program_id = program_id_map[program]

    layered_record = next(
        record
        for record in metadata_provenance_layers
        if record["relative_path"]
        == file_record["relative_path"]
    )

    source_episode_values = (
        layered_record[
            "source_header_metadata"
        ].get(
            "قسمت",
            [],
        )
    )

    source_episode_label = (
        source_episode_values[0]
        if source_episode_values
        else None
    )

    season_number = extract_season_number(
        file_record["relative_path"]
    )

    if file_record["episode_number"] is not None:
        transcript_unit_key = (
            f"episode-{file_record['episode_number']:03d}"
        )

        transcript_unit_basis = (
            "EPISODE_NUMBER_FROM_FILENAME"
        )

    else:
        transcript_unit_key = "unit-001"

        transcript_unit_basis = (
            "SINGLE_PHYSICAL_FILE_PROGRAM"
        )

    transcript_unit_id = (
        f"{program_id}:{transcript_unit_key}"
    )

    physical_file_id = (
        "transcript-file:"
        + file_record["sha256"]
    )

    identity_inventory.append({
        "program": program,
        "program_id": program_id,
        "transcript_unit_id": (
            transcript_unit_id
        ),
        "transcript_unit_key": (
            transcript_unit_key
        ),
        "transcript_unit_basis": (
            transcript_unit_basis
        ),
        "episode_number": (
            file_record["episode_number"]
        ),
        "source_episode_label": (
            source_episode_label
        ),
        "season_number": season_number,
        "physical_file_id": physical_file_id,
        "relative_path": (
            file_record["relative_path"]
        ),
        "sha256": file_record["sha256"],
    })

program_ids = [
    record["program_id"]
    for record in identity_inventory
]

physical_file_ids = [
    record["physical_file_id"]
    for record in identity_inventory
]

transcript_unit_ids = [
    record["transcript_unit_id"]
    for record in identity_inventory
]

duplicate_physical_file_ids = sorted(
    file_id
    for file_id, count
    in Counter(physical_file_ids).items()
    if count > 1
)

duplicate_transcript_unit_ids = sorted(
    unit_id
    for unit_id, count
    in Counter(transcript_unit_ids).items()
    if count > 1
)

unnumbered_identity_records = [
    record
    for record in identity_inventory
    if record["episode_number"] is None
]

asre_shirin_identity_records = [
    record
    for record in identity_inventory
    if record["program"] == "عصر شیرین"
]

asre_shirin_season_counts = Counter(
    record["season_number"]
    for record in asre_shirin_identity_records
)

asre_shirin_season_1_episodes = sorted(
    record["episode_number"]
    for record in asre_shirin_identity_records
    if record["season_number"] == 1
)

asre_shirin_season_2_episodes = sorted(
    record["episode_number"]
    for record in asre_shirin_identity_records
    if record["season_number"] == 2
)

unexpected_season_records = [
    record
    for record in identity_inventory
    if (
        record["program"] != "عصر شیرین"
        and record["season_number"] is not None
    )
]

identity_model_gate_pass = (
    len(program_id_map) == 13
    and len(set(program_id_map.values())) == 13
    and len(identity_inventory) == 183
    and not duplicate_physical_file_ids
    and not duplicate_transcript_unit_ids
    and len(unnumbered_identity_records) == 4
    and asre_shirin_season_counts.get(1) == 13
    and asre_shirin_season_counts.get(2) == 13
    and asre_shirin_season_1_episodes
    == list(range(1, 14))
    and asre_shirin_season_2_episodes
    == list(range(14, 27))
    and not unexpected_season_records
)

print("TRANSCRIPT IDENTITY MODEL")
print("-" * 80)
print(
    "Programs:",
    len(program_id_map),
)
print(
    "Physical files:",
    len(identity_inventory),
)
print(
    "Unique Physical File IDs:",
    len(set(physical_file_ids)),
)
print(
    "Unique Transcript Unit IDs:",
    len(set(transcript_unit_ids)),
)
print(
    "Duplicate Physical File IDs:",
    duplicate_physical_file_ids
    if duplicate_physical_file_ids
    else "NONE",
)
print(
    "Duplicate Transcript Unit IDs:",
    duplicate_transcript_unit_ids
    if duplicate_transcript_unit_ids
    else "NONE",
)

print("\nUNNUMBERED SINGLE-FILE UNITS")
print("-" * 80)

for record in unnumbered_identity_records:
    print(
        " -",
        record["program"],
        "| unit=",
        record["transcript_unit_id"],
        "| source_episode_label=",
        record["source_episode_label"],
        "| basis=",
        record["transcript_unit_basis"],
    )

print("\nASRE SHIRIN SEASON MODEL")
print("-" * 80)
print(
    "Season counts:",
    dict(sorted(asre_shirin_season_counts.items())),
)
print(
    "Season 1 episodes:",
    asre_shirin_season_1_episodes,
)
print(
    "Season 2 episodes:",
    asre_shirin_season_2_episodes,
)
print(
    "Unexpected Season assignments:",
    len(unexpected_season_records),
)

print()
print(
    "TRANSCRIPT IDENTITY MODEL GATE:",
    "PASS"
    if identity_model_gate_pass
    else "FAIL",
)

TRANSCRIPT IDENTITY MODEL
--------------------------------------------------------------------------------
Programs: 13
Physical files: 183
Unique Physical File IDs: 183
Unique Transcript Unit IDs: 183
Duplicate Physical File IDs: NONE
Duplicate Transcript Unit IDs: NONE

UNNUMBERED SINGLE-FILE UNITS
--------------------------------------------------------------------------------
 - شراکت | unit= program:sherakat:unit-001 | source_episode_label= 1 | basis= SINGLE_PHYSICAL_FILE_PROGRAM
 - فناپ | unit= program:fanap:unit-001 | source_episode_label= رویداد ۲۱ | basis= SINGLE_PHYSICAL_FILE_PROGRAM
 - میزان | unit= program:mizan:unit-001 | source_episode_label= 1 | basis= SINGLE_PHYSICAL_FILE_PROGRAM
 - کتاب باز | unit= program:ketab-baz:unit-001 | source_episode_label= فصل ۵ - قسمت ۶۸ | basis= SINGLE_PHYSICAL_FILE_PROGRAM

ASRE SHIRIN SEASON MODEL
--------------------------------------------------------------------------------
Season counts: {1: 13, 2: 13}
Season 1 episodes: [1, 2, 3, 4, 5

<div style="direction: rtl; white-space: normal; line-height: 1;">

### ساخت Inventory Decision برای ورود به Transcript Ingestion

Identity Model با موفقیت تثبیت شد:

`TRANSCRIPT IDENTITY MODEL GATE: PASS`

اکنون برای هر Physical File باید تصمیم Inventory مشخص شود.

این تصمیم تعیین می‌کند که فایل در Notebook 06 چه وضعیتی خواهد داشت.

وضعیت اصلی فایل‌های Primary:

`ACCEPTED_FOR_INGESTION`

اما پذیرش فایل به معنی آزادبودن تمام پردازش‌ها نیست.

برای هر فایل محدودیت Speaker Handling مربوط به Program خودش نیز همراه آن ثبت می‌شود.

برای مثال:

- Programهای Solo می‌توانند Body Attribution پیش‌فرض به دکتر کمیل رودی داشته باشند.
- Programهای Multi-speaker باید Context و Speaker Labelهای موجود را حفظ کنند.
- Programهای دارای Boundary نامشخص نباید Speaker Identity حدسی دریافت کنند.

همچنین وضعیت حقوقی جدا از وضعیت پذیرش فنی نگه داشته می‌شود.

وجود فایل در Repository، وجود Source URL یا Human-confirmed authorship/speaking evidence به‌تنهایی Rights Evidence نیست.

بنابراین تا زمانی که مدرک مستقل حقوقی وجود نداشته باشد:

`rights_status = EVIDENCE_REQUIRED`

ثبت می‌شود.

برای Episode 21 از `در جستجوی خوشبختی` هیچ فایل واقعی وجود ندارد، بنابراین:

`MISSING_SOURCE`

به‌صورت Gap Record جداگانه حفظ می‌شود.

Candidate مربوط به Site 32 نیز:

`RECOVERY_CANDIDATE_NOT_ACCEPTED`

باقی می‌ماند و وارد Accepted Corpus نمی‌شود.

Review Materialهای دیگر نیز در این مرحله Accepted نمی‌شوند.

**معیار PASS:**

- دقیقاً `183` Physical File وضعیت `ACCEPTED_FOR_INGESTION` داشته باشند.
- هر فایل Speaker Policy Program خودش را دریافت کند.
- همه فایل‌ها `rights_status = EVIDENCE_REQUIRED` داشته باشند.
- Episode 21 به‌صورت `MISSING_SOURCE` باقی بماند.
- Candidate Site 32 Accepted نشود.
- هیچ Review Material به Accepted Corpus اضافه نشود.

</div>

In [17]:
# Code Cell 17

inventory_decision_records = []

for identity_record in identity_inventory:
    program = identity_record["program"]

    speaker_policy = transcript_speaker_policy[
        program
    ]

    decision_record = {
        **identity_record,
        "inventory_decision": (
            "ACCEPTED_FOR_INGESTION"
        ),
        "rights_status": "EVIDENCE_REQUIRED",
        "rights_basis": (
            "NO_INDEPENDENT_RIGHTS_EVIDENCE_RECORDED"
        ),
        "speaker_policy": (
            speaker_policy["policy"]
        ),
        "primary_speaker": (
            speaker_policy["primary_speaker"]
        ),
        "default_body_attribution_allowed": (
            speaker_policy[
                "default_body_attribution_allowed"
            ]
        ),
        "preserve_other_turn_context": (
            speaker_policy[
                "preserve_other_turn_context"
            ]
        ),
        "unknown_speaker_inference_allowed": (
            speaker_policy[
                "unknown_speaker_inference_allowed"
            ]
        ),
    }

    inventory_decision_records.append(
        decision_record
    )

accepted_for_ingestion = [
    record
    for record in inventory_decision_records
    if record["inventory_decision"]
    == "ACCEPTED_FOR_INGESTION"
]

rights_status_counts = Counter(
    record["rights_status"]
    for record in inventory_decision_records
)

speaker_policy_counts = Counter(
    record["speaker_policy"]
    for record in inventory_decision_records
)

unsafe_unknown_inference_records = [
    record
    for record in inventory_decision_records
    if record[
        "unknown_speaker_inference_allowed"
    ]
]

unsafe_multi_speaker_default_attribution = [
    record
    for record in inventory_decision_records
    if (
        record["speaker_policy"]
        != "HUMAN_CONFIRMED_SOLO"
        and record[
            "default_body_attribution_allowed"
        ]
    )
]

inventory_gap_records = [
    {
        **episode_21_inventory_gap,
        "program_id": program_id_map[
            "در جستجوی خوشبختی"
        ],
        "transcript_unit_id": (
            program_id_map[
                "در جستجوی خوشبختی"
            ]
            + ":episode-021"
        ),
        "physical_file_id": None,
        "relative_path": None,
        "rights_status": "NOT_APPLICABLE_NO_SOURCE_FILE",
    }
]

review_candidate_records = [
    {
        **episode_21_recovery_candidate,
        "inventory_decision": (
            "RECOVERY_CANDIDATE_NOT_ACCEPTED"
        ),
        "rights_status": "EVIDENCE_REQUIRED",
    }
]

other_review_material_records = [
    {
        "relative_path": record["relative_path"],
        "inventory_decision": "REVIEW_ONLY_NOT_ACCEPTED",
    }
    for record in review_materials
    if record["relative_path"]
    != episode_21_recovery_candidate[
        "relative_path"
    ]
]

accepted_review_paths = {
    record["relative_path"]
    for record in accepted_for_ingestion
    if "/extracted/" in record["relative_path"]
}

inventory_decision_gate_pass = (
    len(inventory_decision_records) == 183
    and len(accepted_for_ingestion) == 183
    and rights_status_counts
    == {"EVIDENCE_REQUIRED": 183}
    and not unsafe_unknown_inference_records
    and not unsafe_multi_speaker_default_attribution
    and len(inventory_gap_records) == 1
    and inventory_gap_records[0][
        "inventory_status"
    ] == "MISSING_SOURCE"
    and len(review_candidate_records) == 1
    and review_candidate_records[0][
        "inventory_decision"
    ] == "RECOVERY_CANDIDATE_NOT_ACCEPTED"
    and not accepted_review_paths
)

print("TRANSCRIPT INVENTORY DECISION SUMMARY")
print("-" * 80)
print(
    "Accepted Physical Files:",
    len(accepted_for_ingestion),
)
print(
    "Rights status counts:",
    dict(rights_status_counts),
)
print(
    "Unknown-speaker inference violations:",
    len(unsafe_unknown_inference_records),
)
print(
    "Unsafe multi-speaker default attribution:",
    len(
        unsafe_multi_speaker_default_attribution
    ),
)

print("\nSPEAKER POLICY FILE COUNTS")
print("-" * 80)

for policy, count in sorted(
    speaker_policy_counts.items()
):
    print(
        f"{policy}: {count}"
    )

print("\nMISSING SOURCE RECORDS")
print("-" * 80)

for record in inventory_gap_records:
    print(
        " -",
        record["canonical_program"],
        "| episode=",
        record["episode_number"],
        "| status=",
        record["inventory_status"],
        "| physical_file=",
        record["physical_file_id"],
    )

print("\nREVIEW CANDIDATE DECISION")
print("-" * 80)

for record in review_candidate_records:
    print(
        " -",
        record["relative_path"],
        "| decision=",
        record["inventory_decision"],
    )

print(
    "\nOther Review materials not accepted:",
    len(other_review_material_records),
)
print(
    "Accepted paths from /extracted/:",
    len(accepted_review_paths),
)

print()
print(
    "TRANSCRIPT INVENTORY DECISION GATE:",
    "PASS"
    if inventory_decision_gate_pass
    else "FAIL",
)

TRANSCRIPT INVENTORY DECISION SUMMARY
--------------------------------------------------------------------------------
Accepted Physical Files: 183
Rights status counts: {'EVIDENCE_REQUIRED': 183}
Unknown-speaker inference violations: 0
Unsafe multi-speaker default attribution: 0

SPEAKER POLICY FILE COUNTS
--------------------------------------------------------------------------------
EXPLICIT_MULTI_SPEAKER: 77
HUMAN_CONFIRMED_SOLO: 63
KNOWN_PARTICIPANTS_UNLABELED_TURNS: 1
MIXED_EXPLICIT_AND_CONTEXTUAL: 38
NARRATIVE_AUDIENCE_CUES: 1
PRIMARY_SPEAKER_KNOWN_BOUNDARIES_UNRESOLVED: 3

MISSING SOURCE RECORDS
--------------------------------------------------------------------------------
 - در جستجوی خوشبختی | episode= 21 | status= MISSING_SOURCE | physical_file= None

REVIEW CANDIDATE DECISION
--------------------------------------------------------------------------------
 - data/transcripts/final/dar-jostojo-final-C/extracted/missing-core/final-c-21_from_site-32.md | decision= RECOVERY_

<div style="direction: rtl; white-space: normal; line-height: 1;">

### ساخت Transcript Inventory Manifest نسخه 1

تمام تصمیم‌های اصلی Transcript Inventory اکنون Gate معتبر دارند.

در این مرحله این اطلاعات در یک Artifact رسمی و قابل Version Control ذخیره می‌شوند:

`data/phase2/manifests/transcript_manifest_v1.json`

Manifest شامل این بخش‌ها خواهد بود:

- Schema و Artifact Version
- فهرست `13` Program
- Program IDها
- Human-confirmed Program Context
- Speaker Handling Policy
- Speaker Aliasهای Human-confirmed
- `183` Physical File پذیرفته‌شده
- SHA-256 هر Physical File
- Transcript Unit ID
- Episode Number در صورت وجود
- Season Number در صورت وجود
- Source Episode Label
- Metadata Provenance Layers
- Inventory Decision
- Rights Status
- Episode 21 Missing Source Record
- Recovery Candidate غیرقابل‌قبول فعلی
- Review Materialهای غیرپذیرفته‌شده

این Manifest فقط Inventory است.

موارد زیر عمداً در آن تولید نمی‌شوند:

- Transcript Segments
- Speaker Turns
- Normalized Text
- Deduplicated Text
- Chunks
- Embeddings

Manifest باید Deterministic باشد؛ یعنی اجرای دوباره Notebook با Sourceهای یکسان باید همان محتوای منطقی را تولید کند.

فایل Candidate مربوط به Episode 21 نیز فقط به‌عنوان Review Evidence ثبت می‌شود و جزو `accepted_files` نخواهد بود.

**معیار PASS:**

- Manifest با Version `v1` ساخته شود.
- دقیقاً `13` Program داشته باشد.
- دقیقاً `183` Accepted Physical File داشته باشد.
- تمام Accepted Fileها SHA-256 یکتا داشته باشند.
- Episode 21 یک `MISSING_SOURCE` Record داشته باشد.
- Candidate مربوط به Site 32 Accepted نباشد.
- تعداد Review Materialها `30` باقی بماند.
- فایل پس از Write دوباره از Disk خوانده و بدون اختلاف Validate شود.

</div>

In [18]:
# Code Cell 18

import json

transcript_manifest_path = (
    project_root
    / "data"
    / "phase2"
    / "manifests"
    / "transcript_manifest_v1.json"
)

transcript_manifest_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

manifest_programs = []

for program in human_confirmed_program_context:
    manifest_programs.append({
        "program": program,
        "program_id": program_id_map[program],
        "human_confirmed_context": (
            human_confirmed_program_context[
                program
            ]
        ),
        "speaker_policy": (
            transcript_speaker_policy[
                program
            ]
        ),
        "speaker_aliases": (
            human_confirmed_speaker_aliases.get(
                program,
                {},
            )
        ),
    })

manifest_accepted_files = []

for decision_record in inventory_decision_records:
    layered_record = next(
        record
        for record in metadata_provenance_layers
        if record["relative_path"]
        == decision_record["relative_path"]
    )

    manifest_accepted_files.append({
        "program": (
            decision_record["program"]
        ),
        "program_id": (
            decision_record["program_id"]
        ),
        "transcript_unit_id": (
            decision_record[
                "transcript_unit_id"
            ]
        ),
        "transcript_unit_key": (
            decision_record[
                "transcript_unit_key"
            ]
        ),
        "transcript_unit_basis": (
            decision_record[
                "transcript_unit_basis"
            ]
        ),
        "episode_number": (
            decision_record[
                "episode_number"
            ]
        ),
        "season_number": (
            decision_record[
                "season_number"
            ]
        ),
        "source_episode_label": (
            decision_record[
                "source_episode_label"
            ]
        ),
        "physical_file_id": (
            decision_record[
                "physical_file_id"
            ]
        ),
        "relative_path": (
            decision_record[
                "relative_path"
            ]
        ),
        "sha256": (
            decision_record["sha256"]
        ),
        "inventory_decision": (
            decision_record[
                "inventory_decision"
            ]
        ),
        "rights_status": (
            decision_record[
                "rights_status"
            ]
        ),
        "rights_basis": (
            decision_record[
                "rights_basis"
            ]
        ),
        "speaker_policy": (
            decision_record[
                "speaker_policy"
            ]
        ),
        "primary_speaker": (
            decision_record[
                "primary_speaker"
            ]
        ),
        "default_body_attribution_allowed": (
            decision_record[
                "default_body_attribution_allowed"
            ]
        ),
        "preserve_other_turn_context": (
            decision_record[
                "preserve_other_turn_context"
            ]
        ),
        "unknown_speaker_inference_allowed": (
            decision_record[
                "unknown_speaker_inference_allowed"
            ]
        ),
        "metadata_layers": {
            "source_header": (
                layered_record[
                    "source_header_metadata"
                ]
            ),
            "source_provenance": (
                layered_record[
                    "source_provenance"
                ]
            ),
            "path_filename_derived": (
                layered_record[
                    "path_filename_derived"
                ]
            ),
            "human_confirmed": (
                layered_record[
                    "human_confirmed"
                ]
            ),
            "source_program_name": (
                layered_record[
                    "source_program_name"
                ]
            ),
            "source_name_variant_status": (
                layered_record[
                    "source_name_variant_status"
                ]
            ),
        },
    })

manifest_review_materials = []

for record in review_materials:
    relative_path = record["relative_path"]

    if (
        relative_path
        == episode_21_recovery_candidate[
            "relative_path"
        ]
    ):
        decision = (
            "RECOVERY_CANDIDATE_NOT_ACCEPTED"
        )
    else:
        decision = "REVIEW_ONLY_NOT_ACCEPTED"

    manifest_review_materials.append({
        "relative_path": relative_path,
        "filename": record["filename"],
        "inventory_decision": decision,
    })

transcript_manifest_v1 = {
    "artifact_type": (
        "phase2_transcript_inventory"
    ),
    "artifact_version": "v1",
    "schema_version": (
        "transcript_inventory_v1"
    ),
    "scope": {
        "notebook": (
            "notebooks/phase2/"
            "05_transcript_inventory.ipynb"
        ),
        "transcript_root": (
            "data/transcripts/final"
        ),
        "inventory_only": True,
        "transcript_ingestion_performed": False,
        "normalization_performed": False,
        "deduplication_performed": False,
        "chunking_performed": False,
    },
    "counts": {
        "programs": len(manifest_programs),
        "accepted_physical_files": (
            len(manifest_accepted_files)
        ),
        "missing_source_records": (
            len(inventory_gap_records)
        ),
        "review_materials": (
            len(manifest_review_materials)
        ),
        "recovery_candidates_not_accepted": (
            sum(
                record[
                    "inventory_decision"
                ]
                == "RECOVERY_CANDIDATE_NOT_ACCEPTED"
                for record
                in manifest_review_materials
            )
        ),
    },
    "programs": manifest_programs,
    "accepted_files": (
        manifest_accepted_files
    ),
    "missing_sources": (
        inventory_gap_records
    ),
    "review_materials": (
        manifest_review_materials
    ),
    "source_name_variants": (
        program_source_name_variants
    ),
    "global_policies": {
        "unknown_speaker_inference_allowed": False,
        "question_response_context_preservation": (
            "REQUIRED_WHEN_MULTI_SPEAKER"
        ),
        "rights_inference_from_repository_presence": (
            "FORBIDDEN"
        ),
        "rights_default_status": (
            "EVIDENCE_REQUIRED"
        ),
        "missing_metadata_policy": (
            "DO_NOT_INVENT"
        ),
    },
}

serialized_manifest = json.dumps(
    transcript_manifest_v1,
    ensure_ascii=False,
    indent=2,
    sort_keys=True,
)

transcript_manifest_path.write_text(
    serialized_manifest + "\n",
    encoding="utf-8",
)

reloaded_manifest = json.loads(
    transcript_manifest_path.read_text(
        encoding="utf-8"
    )
)

accepted_hashes = [
    record["sha256"]
    for record
    in reloaded_manifest[
        "accepted_files"
    ]
]

accepted_paths = {
    record["relative_path"]
    for record
    in reloaded_manifest[
        "accepted_files"
    ]
}

review_paths = {
    record["relative_path"]
    for record
    in reloaded_manifest[
        "review_materials"
    ]
}

episode_21_manifest_records = [
    record
    for record
    in reloaded_manifest[
        "missing_sources"
    ]
    if (
        record["canonical_program"]
        == "در جستجوی خوشبختی"
        and record["episode_number"] == 21
    )
]

recovery_candidate_manifest_records = [
    record
    for record
    in reloaded_manifest[
        "review_materials"
    ]
    if record["inventory_decision"]
    == "RECOVERY_CANDIDATE_NOT_ACCEPTED"
]

manifest_gate_pass = (
    reloaded_manifest[
        "artifact_version"
    ] == "v1"
    and reloaded_manifest[
        "schema_version"
    ] == "transcript_inventory_v1"
    and len(
        reloaded_manifest["programs"]
    ) == 13
    and len(
        reloaded_manifest[
            "accepted_files"
        ]
    ) == 183
    and len(set(accepted_hashes)) == 183
    and len(
        reloaded_manifest[
            "review_materials"
        ]
    ) == 30
    and len(
        episode_21_manifest_records
    ) == 1
    and episode_21_manifest_records[0][
        "inventory_status"
    ] == "MISSING_SOURCE"
    and len(
        recovery_candidate_manifest_records
    ) == 1
    and recovery_candidate_manifest_records[0][
        "relative_path"
    ] not in accepted_paths
    and not (
        accepted_paths
        & review_paths
    )
    and reloaded_manifest
    == transcript_manifest_v1
)

print("TRANSCRIPT INVENTORY MANIFEST V1")
print("-" * 80)
print(
    "Path:",
    transcript_manifest_path.relative_to(
        project_root
    ),
)
print(
    "Programs:",
    len(
        reloaded_manifest["programs"]
    ),
)
print(
    "Accepted Physical Files:",
    len(
        reloaded_manifest[
            "accepted_files"
        ]
    ),
)
print(
    "Unique accepted hashes:",
    len(set(accepted_hashes)),
)
print(
    "Missing Source records:",
    len(
        reloaded_manifest[
            "missing_sources"
        ]
    ),
)
print(
    "Review materials:",
    len(
        reloaded_manifest[
            "review_materials"
        ]
    ),
)
print(
    "Recovery candidates not accepted:",
    len(
        recovery_candidate_manifest_records
    ),
)
print(
    "Accepted / Review path overlap:",
    len(
        accepted_paths
        & review_paths
    ),
)

print("\nPROCESSING BOUNDARIES")
print("-" * 80)

for key, value in (
    reloaded_manifest[
        "scope"
    ].items()
):
    print(
        f"{key}: {value}"
    )

print()
print(
    "TRANSCRIPT INVENTORY MANIFEST GATE:",
    "PASS"
    if manifest_gate_pass
    else "FAIL",
)

TRANSCRIPT INVENTORY MANIFEST V1
--------------------------------------------------------------------------------
Path: data/phase2/manifests/transcript_manifest_v1.json
Programs: 13
Accepted Physical Files: 183
Unique accepted hashes: 183
Missing Source records: 1
Review materials: 30
Recovery candidates not accepted: 1
Accepted / Review path overlap: 0

PROCESSING BOUNDARIES
--------------------------------------------------------------------------------
chunking_performed: False
deduplication_performed: False
inventory_only: True
normalization_performed: False
notebook: notebooks/phase2/05_transcript_inventory.ipynb
transcript_ingestion_performed: False
transcript_root: data/transcripts/final

TRANSCRIPT INVENTORY MANIFEST GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### ممیزی مستقل Transcript Inventory Manifest از روی Disk

Manifest نسخه 1 با موفقیت ساخته شد:

`TRANSCRIPT INVENTORY MANIFEST GATE: PASS`

اما پذیرش نهایی Notebook نباید فقط به متغیرهای Runtime متکی باشد.

در این مرحله Manifest دوباره مستقیماً از Disk خوانده می‌شود و Source Fileهای ثبت‌شده نیز مستقل از متغیرهای قبلی بررسی می‌شوند.

این Audit موارد زیر را کنترل می‌کند:

- Manifest واقعاً روی Disk وجود داشته باشد.
- JSON بدون Error قابل خواندن باشد.
- `13` Program ثبت شده باشد.
- `183` Accepted Physical File ثبت شده باشد.
- تمام Accepted Fileها هنوز روی Disk موجود باشند.
- SHA-256 فعلی هر فایل با Hash ذخیره‌شده در Manifest یکسان باشد.
- هیچ Accepted Path با Review Material هم‌پوشانی نداشته باشد.
- تمام Review Materialهای ثبت‌شده روی Disk موجود باشند.
- Episode 21 مجموعه `در جستجوی خوشبختی` همچنان `MISSING_SOURCE` باشد.
- Candidate مربوط به Site 32 همچنان Accepted نشده باشد.
- `عصر شیرین` همچنان `26` فایل با Seasonهای صحیح داشته باشد.
- هیچ Processing خارج از Scope این Notebook انجام نشده باشد.

نتیجه Audit در فایل Generated Report زیر ذخیره می‌شود:

`reports/phase2/transcript_inventory_acceptance_v1.json`

این Report یک Artifact ارزیابی است و Manifest اصلی را تغییر نمی‌دهد.

**معیار PASS:**

تمام Validationهای مستقل بدون Error و بدون Hash Mismatch عبور کنند.

</div>

In [19]:
# Code Cell 19

import hashlib
import json
from collections import Counter
from pathlib import Path

manifest_path_disk = (
    project_root
    / "data"
    / "phase2"
    / "manifests"
    / "transcript_manifest_v1.json"
)

acceptance_report_path = (
    project_root
    / "reports"
    / "phase2"
    / "transcript_inventory_acceptance_v1.json"
)

acceptance_report_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

disk_manifest = json.loads(
    manifest_path_disk.read_text(
        encoding="utf-8"
    )
)

disk_checks = {}

disk_checks[
    "manifest_exists"
] = manifest_path_disk.is_file()

disk_checks[
    "artifact_type_valid"
] = (
    disk_manifest.get("artifact_type")
    == "phase2_transcript_inventory"
)

disk_checks[
    "artifact_version_valid"
] = (
    disk_manifest.get("artifact_version")
    == "v1"
)

disk_checks[
    "schema_version_valid"
] = (
    disk_manifest.get("schema_version")
    == "transcript_inventory_v1"
)

disk_checks[
    "program_count_valid"
] = (
    len(
        disk_manifest.get(
            "programs",
            [],
        )
    )
    == 13
)

accepted_disk_records = disk_manifest.get(
    "accepted_files",
    [],
)

review_disk_records = disk_manifest.get(
    "review_materials",
    [],
)

missing_disk_records = disk_manifest.get(
    "missing_sources",
    [],
)

disk_checks[
    "accepted_file_count_valid"
] = (
    len(accepted_disk_records)
    == 183
)

disk_checks[
    "review_material_count_valid"
] = (
    len(review_disk_records)
    == 30
)

accepted_missing_files = []
accepted_hash_mismatches = []
accepted_empty_files = []

for record in accepted_disk_records:
    path = (
        project_root
        / record["relative_path"]
    )

    if not path.is_file():
        accepted_missing_files.append(
            record["relative_path"]
        )
        continue

    raw_bytes = path.read_bytes()

    if not raw_bytes:
        accepted_empty_files.append(
            record["relative_path"]
        )

    actual_hash = hashlib.sha256(
        raw_bytes
    ).hexdigest()

    if actual_hash != record["sha256"]:
        accepted_hash_mismatches.append({
            "relative_path": (
                record["relative_path"]
            ),
            "manifest_sha256": (
                record["sha256"]
            ),
            "actual_sha256": actual_hash,
        })

disk_checks[
    "all_accepted_files_exist"
] = not accepted_missing_files

disk_checks[
    "all_accepted_files_nonempty"
] = not accepted_empty_files

disk_checks[
    "all_accepted_hashes_match"
] = not accepted_hash_mismatches

accepted_paths_disk = {
    record["relative_path"]
    for record in accepted_disk_records
}

review_paths_disk = {
    record["relative_path"]
    for record in review_disk_records
}

disk_checks[
    "accepted_review_overlap_absent"
] = not (
    accepted_paths_disk
    & review_paths_disk
)

review_missing_files = [
    relative_path
    for relative_path in review_paths_disk
    if not (
        project_root
        / relative_path
    ).is_file()
]

disk_checks[
    "all_review_files_exist"
] = not review_missing_files

episode_21_disk_records = [
    record
    for record in missing_disk_records
    if (
        record.get(
            "canonical_program"
        )
        == "در جستجوی خوشبختی"
        and record.get(
            "episode_number"
        )
        == 21
    )
]

disk_checks[
    "episode_21_missing_source_valid"
] = (
    len(episode_21_disk_records)
    == 1
    and episode_21_disk_records[0].get(
        "inventory_status"
    )
    == "MISSING_SOURCE"
    and episode_21_disk_records[0].get(
        "physical_file_id"
    )
    is None
)

recovery_candidates_disk = [
    record
    for record in review_disk_records
    if record.get(
        "inventory_decision"
    )
    == "RECOVERY_CANDIDATE_NOT_ACCEPTED"
]

disk_checks[
    "single_recovery_candidate_valid"
] = (
    len(recovery_candidates_disk)
    == 1
    and recovery_candidates_disk[0][
        "relative_path"
    ]
    not in accepted_paths_disk
)

program_file_counts_disk = Counter(
    record["program"]
    for record in accepted_disk_records
)

disk_checks[
    "program_file_counts_total_valid"
] = (
    sum(
        program_file_counts_disk.values()
    )
    == 183
    and len(
        program_file_counts_disk
    )
    == 13
)

asre_records_disk = [
    record
    for record in accepted_disk_records
    if record["program"] == "عصر شیرین"
]

asre_season_counts_disk = Counter(
    record["season_number"]
    for record in asre_records_disk
)

asre_episode_numbers_disk = sorted(
    record["episode_number"]
    for record in asre_records_disk
)

disk_checks[
    "asre_shirin_structure_valid"
] = (
    len(asre_records_disk) == 26
    and asre_season_counts_disk
    == {1: 13, 2: 13}
    and asre_episode_numbers_disk
    == list(range(1, 27))
)

scope_disk = disk_manifest.get(
    "scope",
    {},
)

disk_checks[
    "inventory_scope_preserved"
] = (
    scope_disk.get(
        "inventory_only"
    )
    is True
    and scope_disk.get(
        "transcript_ingestion_performed"
    )
    is False
    and scope_disk.get(
        "normalization_performed"
    )
    is False
    and scope_disk.get(
        "deduplication_performed"
    )
    is False
    and scope_disk.get(
        "chunking_performed"
    )
    is False
)

rights_statuses_disk = Counter(
    record.get(
        "rights_status"
    )
    for record in accepted_disk_records
)

disk_checks[
    "rights_status_policy_valid"
] = (
    rights_statuses_disk
    == {"EVIDENCE_REQUIRED": 183}
)

unknown_speaker_violations_disk = [
    record["relative_path"]
    for record in accepted_disk_records
    if record.get(
        "unknown_speaker_inference_allowed"
    )
    is not False
]

disk_checks[
    "unknown_speaker_inference_forbidden"
] = not unknown_speaker_violations_disk

all_disk_checks_pass = all(
    disk_checks.values()
)

acceptance_report = {
    "artifact_type": (
        "phase2_transcript_inventory_acceptance"
    ),
    "artifact_version": "v1",
    "manifest": str(
        manifest_path_disk.relative_to(
            project_root
        )
    ),
    "result": (
        "PASS"
        if all_disk_checks_pass
        else "FAIL"
    ),
    "checks": disk_checks,
    "counts": {
        "programs": len(
            disk_manifest.get(
                "programs",
                [],
            )
        ),
        "accepted_files": len(
            accepted_disk_records
        ),
        "review_materials": len(
            review_disk_records
        ),
        "missing_sources": len(
            missing_disk_records
        ),
    },
    "exceptions": {
        "accepted_missing_files": (
            accepted_missing_files
        ),
        "accepted_empty_files": (
            accepted_empty_files
        ),
        "accepted_hash_mismatches": (
            accepted_hash_mismatches
        ),
        "review_missing_files": (
            review_missing_files
        ),
        "unknown_speaker_violations": (
            unknown_speaker_violations_disk
        ),
    },
}

acceptance_report_path.write_text(
    json.dumps(
        acceptance_report,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

reloaded_acceptance_report = json.loads(
    acceptance_report_path.read_text(
        encoding="utf-8"
    )
)

acceptance_gate_pass = (
    reloaded_acceptance_report[
        "result"
    ]
    == "PASS"
    and all(
        reloaded_acceptance_report[
            "checks"
        ].values()
    )
)

print("TRANSCRIPT INVENTORY INDEPENDENT ACCEPTANCE")
print("-" * 80)
print(
    "Manifest:",
    manifest_path_disk.relative_to(
        project_root
    ),
)
print(
    "Acceptance report:",
    acceptance_report_path.relative_to(
        project_root
    ),
)
print(
    "Programs:",
    len(
        disk_manifest["programs"]
    ),
)
print(
    "Accepted files:",
    len(accepted_disk_records),
)
print(
    "Review materials:",
    len(review_disk_records),
)
print(
    "Missing sources:",
    len(missing_disk_records),
)

print("\nINDEPENDENT CHECKS")
print("-" * 80)

for check_name, result in (
    reloaded_acceptance_report[
        "checks"
    ].items()
):
    print(
        f"{check_name}:",
        "PASS" if result else "FAIL",
    )

print("\nEXCEPTIONS")
print("-" * 80)
print(
    "Missing accepted files:",
    len(accepted_missing_files),
)
print(
    "Empty accepted files:",
    len(accepted_empty_files),
)
print(
    "Hash mismatches:",
    len(accepted_hash_mismatches),
)
print(
    "Missing Review files:",
    len(review_missing_files),
)
print(
    "Unknown-speaker violations:",
    len(
        unknown_speaker_violations_disk
    ),
)

print()
print(
    "TRANSCRIPT INVENTORY INDEPENDENT ACCEPTANCE GATE:",
    "PASS"
    if acceptance_gate_pass
    else "FAIL",
)

TRANSCRIPT INVENTORY INDEPENDENT ACCEPTANCE
--------------------------------------------------------------------------------
Manifest: data/phase2/manifests/transcript_manifest_v1.json
Acceptance report: reports/phase2/transcript_inventory_acceptance_v1.json
Programs: 13
Accepted files: 183
Review materials: 30
Missing sources: 1

INDEPENDENT CHECKS
--------------------------------------------------------------------------------
accepted_file_count_valid: PASS
accepted_review_overlap_absent: PASS
all_accepted_files_exist: PASS
all_accepted_files_nonempty: PASS
all_accepted_hashes_match: PASS
all_review_files_exist: PASS
artifact_type_valid: PASS
artifact_version_valid: PASS
asre_shirin_structure_valid: PASS
episode_21_missing_source_valid: PASS
inventory_scope_preserved: PASS
manifest_exists: PASS
program_count_valid: PASS
program_file_counts_total_valid: PASS
review_material_count_valid: PASS
rights_status_policy_valid: PASS
schema_version_valid: PASS
single_recovery_candidate_valid: 

<div style="direction: rtl; white-space: normal; line-height: 1;">

### Final Acceptance و Health Check نهایی Notebook 05

Transcript Inventory تا مرحله Independent Acceptance تکمیل شده است.

این Cell آخر، سلامت نهایی Notebook و Artifactهای ساخته‌شده را مستقیماً از روی Disk بررسی می‌کند.

Artifactهای اصلی:

- `data/phase2/manifests/transcript_manifest_v1.json`
- `reports/phase2/transcript_inventory_acceptance_v1.json`

ساختار مورد انتظار Notebook:

- `21` Markdown Cell
- `20` Code Cell
- مجموعاً `41` Cell
- Code Cellهای پیوسته از `01` تا `20`

Final Health Check کنترل می‌کند که:

- Manifest و Acceptance Report معتبر باشند.
- `13` Program ثبت شده باشد.
- `183` فایل Transcript پذیرفته شده باشد.
- `30` Review Material جدا باقی مانده باشد.
- Episode 21 مجموعه `در جستجوی خوشبختی` همچنان `MISSING_SOURCE` باشد.
- Recovery Candidate مربوط به Site 32 وارد Accepted Corpus نشده باشد.
- Rights Status فایل‌های پذیرفته‌شده `EVIDENCE_REQUIRED` باشد.
- Unknown Speaker Inference ممنوع باقی مانده باشد.
- Transcript Ingestion انجام نشده باشد.
- Normalization انجام نشده باشد.
- Deduplication انجام نشده باشد.
- Chunking انجام نشده باشد.
- تمام Code Cellهای `01` تا `19` اجرا شده باشند.
- هیچ Stored Error در Cellهای `01` تا `19` وجود نداشته باشد.
- تمام Code Cellها از نظر Python Syntax معتبر باشند.

پس از PASS این Cell:

`Restart Kernel → Run All`

انجام می‌شود.

اگر پس از Run All همین Final Health Check دوباره PASS باشد، Notebook 05 رسماً Complete خواهد بود.

</div>

In [20]:
# Code Cell 20

import ast
import json
import re
from collections import Counter

notebook_file = (
    project_root
    / "notebooks"
    / "phase2"
    / "05_transcript_inventory.ipynb"
)

manifest_file = (
    project_root
    / "data"
    / "phase2"
    / "manifests"
    / "transcript_manifest_v1.json"
)

acceptance_file = (
    project_root
    / "reports"
    / "phase2"
    / "transcript_inventory_acceptance_v1.json"
)

manifest_final = json.loads(
    manifest_file.read_text(
        encoding="utf-8"
    )
)

acceptance_final = json.loads(
    acceptance_file.read_text(
        encoding="utf-8"
    )
)

accepted_final = manifest_final.get(
    "accepted_files",
    [],
)

review_final = manifest_final.get(
    "review_materials",
    [],
)

missing_final = manifest_final.get(
    "missing_sources",
    [],
)

accepted_paths_final = {
    record["relative_path"]
    for record in accepted_final
}

review_paths_final = {
    record["relative_path"]
    for record in review_final
}

recovery_candidates_final = [
    record
    for record in review_final
    if record.get(
        "inventory_decision"
    )
    == "RECOVERY_CANDIDATE_NOT_ACCEPTED"
]

rights_counts_final = Counter(
    record.get("rights_status")
    for record in accepted_final
)

unknown_speaker_violations_final = [
    record["relative_path"]
    for record in accepted_final
    if record.get(
        "unknown_speaker_inference_allowed"
    )
    is not False
]

scope_final = manifest_final.get(
    "scope",
    {},
)

final_artifact_checks = {
    "manifest_exists": (
        manifest_file.is_file()
    ),
    "acceptance_report_exists": (
        acceptance_file.is_file()
    ),
    "artifact_type_valid": (
        manifest_final.get(
            "artifact_type"
        )
        == "phase2_transcript_inventory"
    ),
    "artifact_version_valid": (
        manifest_final.get(
            "artifact_version"
        )
        == "v1"
    ),
    "schema_version_valid": (
        manifest_final.get(
            "schema_version"
        )
        == "transcript_inventory_v1"
    ),
    "programs_13": (
        len(
            manifest_final.get(
                "programs",
                [],
            )
        )
        == 13
    ),
    "accepted_files_183": (
        len(accepted_final) == 183
    ),
    "review_materials_30": (
        len(review_final) == 30
    ),
    "missing_sources_1": (
        len(missing_final) == 1
    ),
    "episode_21_missing_source": (
        len(missing_final) == 1
        and missing_final[0].get(
            "canonical_program"
        )
        == "در جستجوی خوشبختی"
        and missing_final[0].get(
            "episode_number"
        )
        == 21
        and missing_final[0].get(
            "inventory_status"
        )
        == "MISSING_SOURCE"
        and missing_final[0].get(
            "physical_file_id"
        )
        is None
    ),
    "single_recovery_candidate": (
        len(
            recovery_candidates_final
        )
        == 1
    ),
    "recovery_candidate_not_accepted": (
        len(
            recovery_candidates_final
        )
        == 1
        and recovery_candidates_final[0][
            "relative_path"
        ]
        not in accepted_paths_final
    ),
    "accepted_review_overlap_absent": (
        not (
            accepted_paths_final
            & review_paths_final
        )
    ),
    "rights_policy_valid": (
        rights_counts_final
        == {"EVIDENCE_REQUIRED": 183}
    ),
    "unknown_speaker_inference_forbidden": (
        not unknown_speaker_violations_final
    ),
    "inventory_only": (
        scope_final.get(
            "inventory_only"
        )
        is True
    ),
    "ingestion_not_performed": (
        scope_final.get(
            "transcript_ingestion_performed"
        )
        is False
    ),
    "normalization_not_performed": (
        scope_final.get(
            "normalization_performed"
        )
        is False
    ),
    "deduplication_not_performed": (
        scope_final.get(
            "deduplication_performed"
        )
        is False
    ),
    "chunking_not_performed": (
        scope_final.get(
            "chunking_performed"
        )
        is False
    ),
    "independent_acceptance_pass": (
        acceptance_final.get(
            "result"
        )
        == "PASS"
        and all(
            acceptance_final.get(
                "checks",
                {},
            ).values()
        )
    ),
}

final_artifacts_pass = all(
    final_artifact_checks.values()
)

notebook_data = json.loads(
    notebook_file.read_text(
        encoding="utf-8"
    )
)

cells = notebook_data.get(
    "cells",
    [],
)

markdown_cells = [
    cell
    for cell in cells
    if cell.get("cell_type")
    == "markdown"
]

code_cells = [
    cell
    for cell in cells
    if cell.get("cell_type")
    == "code"
]

code_number_pattern = re.compile(
    r"^\s*#\s*Code Cell\s+(\d+)\s*$",
    re.IGNORECASE,
)

numbered_code_cells = []
numbered_cell_map = {}
syntax_errors = []

for index, cell in enumerate(
    code_cells,
    start=1,
):
    source = "".join(
        cell.get(
            "source",
            [],
        )
    )

    source_lines = source.splitlines()

    first_line = (
        source_lines[0]
        if source_lines
        else ""
    )

    match = code_number_pattern.match(
        first_line
    )

    if match:
        code_number = int(
            match.group(1)
        )

        numbered_code_cells.append(
            code_number
        )

        numbered_cell_map[
            code_number
        ] = cell

    try:
        ast.parse(source)
    except SyntaxError as exc:
        syntax_errors.append({
            "code_cell_index": index,
            "message": str(exc),
        })

expected_numbers = list(
    range(1, 21)
)

previous_expected_numbers = list(
    range(1, 20)
)

executed_previous_numbers = [
    number
    for number in previous_expected_numbers
    if (
        number in numbered_cell_map
        and numbered_cell_map[
            number
        ].get(
            "execution_count"
        )
        is not None
    )
]

unexecuted_previous_numbers = [
    number
    for number in previous_expected_numbers
    if (
        number not in numbered_cell_map
        or numbered_cell_map[
            number
        ].get(
            "execution_count"
        )
        is None
    )
]

stored_error_previous_numbers = []

for number in previous_expected_numbers:
    cell = numbered_cell_map.get(
        number
    )

    if cell is None:
        continue

    outputs = cell.get(
        "outputs",
        [],
    )

    if any(
        output.get(
            "output_type"
        )
        == "error"
        for output in outputs
    ):
        stored_error_previous_numbers.append(
            number
        )

duplicate_numbers = sorted({
    number
    for number
    in numbered_code_cells
    if numbered_code_cells.count(
        number
    )
    > 1
})

missing_numbers = sorted(
    set(expected_numbers)
    - set(numbered_code_cells)
)

notebook_structure_checks = {
    "notebook_exists": (
        notebook_file.is_file()
    ),
    "total_cells_41": (
        len(cells) == 41
    ),
    "markdown_cells_21": (
        len(markdown_cells) == 21
    ),
    "code_cells_20": (
        len(code_cells) == 20
    ),
    "numbered_code_cells_20": (
        len(
            numbered_code_cells
        )
        == 20
    ),
    "sequential_numbering": (
        numbered_code_cells
        == expected_numbers
    ),
    "no_duplicate_numbers": (
        not duplicate_numbers
    ),
    "no_missing_numbers": (
        not missing_numbers
    ),
    "no_syntax_errors": (
        not syntax_errors
    ),
    "previous_19_executed": (
        executed_previous_numbers
        == previous_expected_numbers
        and not unexecuted_previous_numbers
    ),
    "previous_19_without_stored_errors": (
        not stored_error_previous_numbers
    ),
}

notebook_structure_pass = all(
    notebook_structure_checks.values()
)

runtime_final_acceptance = (
    final_artifacts_pass
    and notebook_structure_pass
)

print("NOTEBOOK 05 FINAL HEALTH CHECK")
print("-" * 80)
print(
    "Notebook:",
    notebook_file.relative_to(
        project_root
    ),
)
print(
    "Manifest:",
    manifest_file.relative_to(
        project_root
    ),
)
print(
    "Acceptance report:",
    acceptance_file.relative_to(
        project_root
    ),
)

print("\nFINAL ARTIFACT CHECKS")
print("-" * 80)

for check_name, result in (
    final_artifact_checks.items()
):
    print(
        f"{check_name}:",
        "PASS"
        if result
        else "FAIL",
    )

print("\nNOTEBOOK STRUCTURE CHECKS")
print("-" * 80)

for check_name, result in (
    notebook_structure_checks.items()
):
    print(
        f"{check_name}:",
        "PASS"
        if result
        else "FAIL",
    )

print("\nNOTEBOOK HEALTH")
print("-" * 80)
print(
    "Total cells:",
    len(cells),
)
print(
    "Markdown cells:",
    len(markdown_cells),
)
print(
    "Code cells:",
    len(code_cells),
)
print(
    "Executed previous code cells:",
    len(
        executed_previous_numbers
    ),
)
print(
    "Unexecuted previous code cells:",
    len(
        unexecuted_previous_numbers
    ),
)
print(
    "Stored error previous cells:",
    len(
        stored_error_previous_numbers
    ),
)
print(
    "Syntax errors:",
    len(syntax_errors),
)
print(
    "Numbered code cells:",
    len(
        numbered_code_cells
    ),
)
print(
    "Sequential numbering:",
    (
        "PASS"
        if numbered_code_cells
        == expected_numbers
        else "FAIL"
    ),
)
print(
    "Duplicate numbers:",
    (
        duplicate_numbers
        if duplicate_numbers
        else "NONE"
    ),
)
print(
    "Missing numbers:",
    (
        missing_numbers
        if missing_numbers
        else "NONE"
    ),
)

if unexecuted_previous_numbers:
    print(
        "\nUNEXECUTED PREVIOUS CODE CELLS:",
        unexecuted_previous_numbers,
    )

if stored_error_previous_numbers:
    print(
        "\nSTORED ERROR PREVIOUS CELLS:",
        stored_error_previous_numbers,
    )

if syntax_errors:
    print("\nSYNTAX ERRORS")

    for item in syntax_errors:
        print(
            " - Code Cell",
            item[
                "code_cell_index"
            ],
            ":",
            item["message"],
        )

print("\nFINAL STATUS")
print("-" * 80)
print(
    "Final artifacts:",
    "PASS"
    if final_artifacts_pass
    else "FAIL",
)
print(
    "Notebook structure:",
    "PASS"
    if notebook_structure_pass
    else "FAIL",
)
print(
    "Final acceptance marker:",
    "PASS"
    if runtime_final_acceptance
    else "FAIL",
)

print()
print(
    "NOTEBOOK 05 HEALTH:",
    "PASS"
    if runtime_final_acceptance
    else "FAIL",
)

NOTEBOOK 05 FINAL HEALTH CHECK
--------------------------------------------------------------------------------
Notebook: notebooks/phase2/05_transcript_inventory.ipynb
Manifest: data/phase2/manifests/transcript_manifest_v1.json
Acceptance report: reports/phase2/transcript_inventory_acceptance_v1.json

FINAL ARTIFACT CHECKS
--------------------------------------------------------------------------------
manifest_exists: PASS
acceptance_report_exists: PASS
artifact_type_valid: PASS
artifact_version_valid: PASS
schema_version_valid: PASS
programs_13: PASS
accepted_files_183: PASS
review_materials_30: PASS
missing_sources_1: PASS
episode_21_missing_source: PASS
single_recovery_candidate: PASS
recovery_candidate_not_accepted: PASS
accepted_review_overlap_absent: PASS
rights_policy_valid: PASS
unknown_speaker_inference_forbidden: PASS
inventory_only: PASS
ingestion_not_performed: PASS
normalization_not_performed: PASS
deduplication_not_performed: PASS
chunking_not_performed: PASS
independen